# Geometry ↔ Colour Mutual Denoising — a controlled feasibility study

**Long-term goal.** A *joint* 6-D diffusion model for colored point-cloud completion, where a point is
$p_i=[x_i,y_i,z_i,r_i,g_i,b_i]$.

**Hypothesis under test here (and *only* this).** Geometry and colour carry mutually useful
information *during denoising*. Two symmetric, controlled experiments:

| | noised | conditioned on | baseline | conditioned model | question |
|---|---|---|---|---|---|
| **Experiment 1** | geometry $G_t$ | clean colour $C_0$ | `Model G` $\hat\epsilon_g=f_G(G_t,t)$ | `Model G+C` $\hat\epsilon_g=f_{G+C}(G_t,C_0,t)$ | is $\mathrm{MSE}_{G+C}<\mathrm{MSE}_{G}$ ? |
| **Experiment 2** | colour $C_t$ | clean geometry $G_0$ | `Model C` $\hat\epsilon_c=f_C(C_t,t)$ | `Model C+G` $\hat\epsilon_c=f_{C+G}(C_t,G_0,t)$ | is $\mathrm{MSE}_{C+G}<\mathrm{MSE}_{C}$ ? |

**This is NOT** a completion system, not PDR / CGNet / RFNet, not a reverse-diffusion sampler, not
part-aware modelling, and there is no cross-attention. Nothing here establishes that a joint
geometry+colour completion model beats a geometry-only one. It only asks whether one modality is
*predictive of* the other under corruption — the cheapest possible precondition for the joint model
being worth building.

**Design rules honoured throughout (see §10 and §11):**

1. **Colour is an attribute, never a coordinate.** No kNN / FPS / ball-query / graph is ever built in a
   mixed $\|XYZ_i-XYZ_j\|^2+\|RGB_i-RGB_j\|^2$ space. Every neighbourhood in this notebook is built
   from XYZ only, and §10 runs an executable **leakage test** proving `Model C` cannot see geometry.
2. **Fair pairs.** The two models inside a pair share the backbone, width, depth, timestep mechanism,
   optimiser, schedule, batch order, sampled timesteps *and* the sampled noise. They differ in exactly
   one thing: 3 extra input channels. Parameter counts are printed in §10.
3. **Paired evaluation.** At every evaluated timestep both models in a pair see byte-identical
   $G_t$ / $C_t$, produced from a fixed seed. Differences are conditioning, not luck.
4. **No part labels.** The dataset ships per-point ShapeNet-Part labels; they are deliberately
   *never loaded into any model*. Part-consistency is only the motivation for the hypothesis, not an
   input. (§11 uses an XYZ-only FPS-Voronoi partition instead when it needs spatial regions.)

**How to run this on Kaggle**

1. *Settings → Accelerator →* **GPU** (T4/P100). CPU also works for §12 but is slow for §14–17.
2. *Settings → Internet →* **ON** (needed for the HuggingFace dataset download in §3).
3. Run §1 → §12. Read the sanity-check verdict.
4. Only if it says `SANITY CHECK PASSED`, set `RUN_FULL_EXPERIMENT = True` in §1, re-run §1, then run
   §13 → §25.
5. No Kaggle Secrets, API keys or private datasets are required. The dataset is public.

Nothing in this notebook is pre-computed. Every number, table, figure and sentence of the final report
in §24 is generated from *your* run.

## 1 · Configuration

The only cell you normally edit. Everything downstream reads from here.

**If you hit CUDA OOM, reduce in this order:** `NUM_POINTS` (1024 → 512) → `BATCH_SIZE` (8 → 4) →
`EVAL_REPEATS` (4 → 2) → `WIDTH` (128 → 64). The models are tiny (≈0.3 M params); the point count
dominates memory.

**If it is too slow:** drop `NUM_EPOCHS` to 200 and `NUM_TRAIN_SHAPES` to 40 — the trend is usually
already visible.

In [ ]:
import os

# Must be set BEFORE torch initialises CUDA, so it lives at the very top of the config cell.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# ─────────────────────────── core experiment ───────────────────────────
SEED             = 0
CATEGORY         = "chair"      # "airplane" | "car" | "chair"  (one category, as specified)
NUM_POINTS       = 1024         # points per cloud; source clouds hold 8192
NUM_TRAIN_SHAPES = 80           # 20-100 is the sane range for a first run
NUM_VAL_SHAPES   = 20           # strictly disjoint objects (asserted in §6)
BATCH_SIZE       = 8
NUM_EPOCHS       = 600          # 80 shapes / bs 8 = 10 steps/epoch -> 6000 steps per model
LEARNING_RATE    = 1e-3         # AdamW; cosine-decayed to LEARNING_RATE/100
WEIGHT_DECAY     = 1e-4
LR_SCHEDULE      = "cosine"     # "cosine" | "constant"
T                = 1000         # diffusion steps
BETA_SCHEDULE    = "cosine"     # "cosine" (Nichol & Dhariwal) | "linear"

# ─────────────────────────── model (identical for all four) ───────────────────────────
WIDTH    = 128
N_BLOCKS = 3
# Optional local (EdgeConv) aggregation. OFF by default. It may ONLY be enabled for the geometry
# pair, where the kNN graph is built from G_t — a tensor BOTH Model G and Model G+C possess, so the
# pair stays fair and no colour enters the neighbourhood. It is structurally impossible to enable
# for the colour pair without either leaking geometry into Model C or breaking the pair; §10 asserts
# this. Colour NEVER participates in any neighbourhood construction.
USE_LOCAL_GEOMETRY_BACKBONE = False
LOCAL_K  = 16

# ─────────────────────────── run control ───────────────────────────
RUN_SANITY_CHECK    = True      # §12; cheap, leave on
SANITY_CLOUDS       = 6         # §12 tiny memorisation set (5-10)
SANITY_STEPS        = 1500      # §12 steps per model; raise to 3000 if a model is borderline
RUN_FULL_EXPERIMENT = False     # <<< flip to True ONLY after §12 prints "SANITY CHECK PASSED"
RESUME              = True      # pick training back up from the per-epoch checkpoint
FORCE_RETRAIN       = False      # ignore existing checkpoints and start over
TIME_BUDGET_MIN_PER_MODEL = None  # e.g. 20 -> stop that model early but still save + evaluate
DETERMINISTIC_STRICT = True      # torch.use_deterministic_algorithms(..., warn_only=True)

# ─────────────────────────── evaluation ───────────────────────────
EVAL_TIMESTEPS   = [50, 100, 250, 500, 750, 900]
EVAL_REPEATS     = 4     # independent noise draws per (shape, t); shared byte-for-byte across a pair
VAL_BANK_REPEATS = 8     # frozen (t, noise) draws per val shape used for the per-epoch val loss
KNN_K            = 8     # k for the local colour-consistency metric  (XYZ neighbourhoods only)
N_REGIONS        = 32    # FPS-Voronoi cells for the region-wise colour metric (XYZ only)

# ─────────────────────────── visualisation ───────────────────────────
N_VIS_SHAPES = 3
VIS_T        = 250       # noise level used for the qualitative panels
VIEW_ELEV, VIEW_AZIM = 22, 135   # identical camera for every 3-D panel in the notebook
PT_SIZE      = 3.0
SAVE_PLOTLY  = True      # also write interactive .html panels (reuses the repo's plotly helper)

# ─────────────────────────── data ───────────────────────────
HF_DATASET      = "eylulpelinkilic/Colored_Point_Clouds"   # PUBLIC, no token needed
DATA_SOURCE     = "auto"       # "auto" | "local" | "hf" | "synthetic"
LOCAL_DATA_ROOT = os.environ.get("PCC_DATA_ROOT", "")      # only used by "local"/"auto"
SUBSAMPLE       = "random"     # "random" (fast) | "fps" (more uniform, ~1 min for 100 shapes)

# ─────────────────────────── repository (optional) ───────────────────────────
# This notebook is SELF-CONTAINED: every function it reuses from the repo is inlined below with a
# comment naming the source file. Cloning is only needed if you want to import repo modules yourself.
CLONE_REPO = False
REPO_URL    = "https://github.com/eylulpelinkilic/Colored_Point_Cloud_Completion.git"
REPO_BRANCH = "Pelin"

# ─────────────────────────── paths ───────────────────────────
ON_KAGGLE   = os.path.isdir("/kaggle/working")
RESULTS_DIR = "/kaggle/working/results" if ON_KAGGLE else os.path.abspath("./results")
CACHE_DIR   = "/kaggle/temp/pcc_cache"  if ON_KAGGLE else os.path.abspath("./.pcc_cache")
REPO_DIR    = "/kaggle/temp/pcc_repo"   if ON_KAGGLE else os.path.abspath("./.pcc_repo")

CKPT_DIR = os.path.join(RESULTS_DIR, "checkpoints")
HIST_DIR = os.path.join(RESULTS_DIR, "history")
TAB_DIR  = os.path.join(RESULTS_DIR, "tables")
FIG_DIR  = os.path.join(RESULTS_DIR, "figures")
for _d in (RESULTS_DIR, CKPT_DIR, HIST_DIR, TAB_DIR, FIG_DIR):
    os.makedirs(_d, exist_ok=True)
try:
    os.makedirs(CACHE_DIR, exist_ok=True)
except OSError:                      # unwritable /kaggle/temp -> fall back
    CACHE_DIR = os.path.abspath("./.pcc_cache"); os.makedirs(CACHE_DIR, exist_ok=True)

try:
    import torch as _t
    DEVICE = "cuda" if _t.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

SANITY_PASSED = None   # set by §12; never assumed

print("ON_KAGGLE   :", ON_KAGGLE)
print("DEVICE      :", DEVICE)
print("RESULTS_DIR :", RESULTS_DIR)
print("CACHE_DIR   :", CACHE_DIR)
print("category    :", CATEGORY, "| points", NUM_POINTS,
      "| shapes", NUM_TRAIN_SHAPES, "train /", NUM_VAL_SHAPES, "val")
print("RUN_FULL_EXPERIMENT =", RUN_FULL_EXPERIMENT,
      "" if RUN_FULL_EXPERIMENT else "  <- §14-17 will be skipped until you flip this")

## 2 · Environment / dependencies

Kaggle already ships torch, numpy, pandas, scipy and matplotlib. Only `huggingface_hub` (and
optionally `plotly`) may be missing, so we install just those, quietly, and only if the import fails.

**Inspect:** the CUDA line. If it says `cuda: False` the notebook still runs, but §14–17 will be slow —
turn the accelerator on in *Settings → Accelerator*.

In [ ]:
import importlib, subprocess, sys

def _ensure(module, pip_name=None):
    try:
        importlib.import_module(module); return "present"
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=False)
        try:
            importlib.import_module(module); return "installed"
        except ImportError:
            return "MISSING"

for _m in ("numpy", "torch", "pandas", "scipy", "matplotlib", "huggingface_hub", "plotly"):
    print(f"  {_m:16s} {_ensure(_m)}")

import numpy as np, torch, pandas as pd, matplotlib, scipy
print()
print("numpy", np.__version__, "| torch", torch.__version__,
      "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__, "| scipy", scipy.__version__)
print("cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only")
if torch.cuda.is_available():
    print("gpu memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
import shutil as _sh
print("free disk : %.0f GB" % (_sh.disk_usage(RESULTS_DIR).free / 1e9))

## 3 · Repository and dataset setup

**Dataset.** `labeled_s3/<synset>/<model>.npz` from the **public** HuggingFace dataset
`eylulpelinkilic/Colored_Point_Clouds` — this project's own derived benchmark (colored ShapeNet GT,
8192 points, real texture). Keys: `xyz (8192,3) float32`, `rgb (8192,3) float32 in [0,1]`,
`part (8192,) int16`. **No token, no Kaggle Secret, no gated licence** — the whole category is ~20 MB.

*The `part` array is loaded only so §5 can report that it exists; it is never given to a model.*

Resolution order for `DATA_SOURCE="auto"`: local `$PCC_DATA_ROOT` → HuggingFace → synthetic fallback.
The **synthetic fallback is a smoke-test only** and is flagged loudly everywhere it would affect a
conclusion; it exists so the notebook stays runnable if Kaggle's internet is off.

**Change this if needed:** `HF_DATASET` (§1) if you publish under a different repo id; `CATEGORY` to
`"airplane"` / `"car"`.

In [ ]:
import glob, numpy as np

SYNSETS = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
assert CATEGORY in SYNSETS, f"CATEGORY must be one of {list(SYNSETS)}"
SYNSET = SYNSETS[CATEGORY]

if CLONE_REPO and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", REPO_BRANCH], check=False)
    if os.path.isdir(REPO_DIR):
        sys.path.insert(0, REPO_DIR); print("repo cloned ->", REPO_DIR)

IS_SYNTHETIC = False
DATA_ORIGIN  = None
FILES        = []


def _local_files():
    if not LOCAL_DATA_ROOT:
        return []
    return sorted(glob.glob(os.path.join(LOCAL_DATA_ROOT, "labeled_s3", SYNSET, "*.npz")))


def _hf_files():
    from huggingface_hub import snapshot_download
    root = snapshot_download(HF_DATASET, repo_type="dataset",
                             allow_patterns=[f"labeled_s3/{SYNSET}/*.npz"],
                             cache_dir=CACHE_DIR)
    return sorted(glob.glob(os.path.join(root, "labeled_s3", SYNSET, "*.npz")))


# --- synthetic fallback -------------------------------------------------------------------
# Adapted from scripts/toy_part_color.py (build_toy / box_surface). Chair-like axis-aligned boxes.
# Each region carries TWO flat materials, mirroring the real finding that a semantic part usually
# holds several flat materials rather than a smooth gradient -> a genuine geometry<->colour relation
# for a smoke test, with a fixed cross-object palette so the relation is learnable.
_TOY_PARTS = [((0.0, 0.90, 0.00), (2.0, 0.15, 2.0)), ((0.0, 1.65, -0.93), (2.0, 1.5, 0.15)),
              ((-0.9, 0.45, -0.9), (0.16, 0.9, 0.16)), ((0.9, 0.45, -0.9), (0.16, 0.9, 0.16)),
              ((-0.9, 0.45, 0.9), (0.16, 0.9, 0.16)), ((0.9, 0.45, 0.9), (0.16, 0.9, 0.16))]
_TOY_PALETTE = np.array([[0.80, 0.22, 0.18], [0.18, 0.35, 0.75], [0.30, 0.30, 0.32],
                         [0.30, 0.30, 0.32], [0.30, 0.30, 0.32], [0.30, 0.30, 0.32],
                         [0.95, 0.88, 0.70], [0.10, 0.55, 0.45], [0.85, 0.85, 0.88],
                         [0.85, 0.85, 0.88], [0.85, 0.85, 0.88], [0.85, 0.85, 0.88]])


def _box_surface(center, size, n, rng):
    center = np.asarray(center, float); hx, hy, hz = np.asarray(size, float) / 2.0
    areas = np.array([hy*hz, hy*hz, hx*hz, hx*hz, hx*hy, hx*hy]) * 4
    counts = rng.multinomial(n, areas / areas.sum()); pts = []
    for face, k in enumerate(counts):
        if k == 0: continue
        u, v = rng.uniform(-1, 1, k), rng.uniform(-1, 1, k); o = np.ones(k)
        p = [np.stack([o, u, v], 1), np.stack([-o, u, v], 1), np.stack([u, o, v], 1),
             np.stack([u, -o, v], 1), np.stack([u, v, o], 1), np.stack([u, v, -o], 1)][face]
        pts.append(p * np.array([hx, hy, hz]) + center)
    return np.concatenate(pts, 0)


def _synthetic_shape(seed, n=8192):
    rng = np.random.default_rng(seed)
    sizes = np.array([s for _, s in _TOY_PARTS], float)
    area = 2 * (sizes[:, 0]*sizes[:, 1] + sizes[:, 1]*sizes[:, 2] + sizes[:, 0]*sizes[:, 2])
    counts = rng.multinomial(n, area / area.sum())
    xyz, rgb, part = [], [], []
    for pid, ((c, s), k) in enumerate(zip(_TOY_PARTS, counts)):
        if k == 0: continue
        c = np.asarray(c, float) * (1 + rng.uniform(-0.10, 0.10, 3))
        s = np.asarray(s, float) * (1 + rng.uniform(-0.10, 0.10, 3))
        p = _box_surface(c, s, k, rng)
        second = p[:, 1] > (c[1])                     # second material on the upper half
        col = np.where(second[:, None], _TOY_PALETTE[pid + 6], _TOY_PALETTE[pid])
        col = col + rng.normal(0, 0.02, (k, 3))
        xyz.append(p); rgb.append(np.clip(col, 0, 1)); part.append(np.full(k, pid))
    return (np.concatenate(xyz).astype(np.float32), np.concatenate(rgb).astype(np.float32),
            np.concatenate(part).astype(np.int16))


_need = NUM_TRAIN_SHAPES + NUM_VAL_SHAPES
_sources = {"auto": ["local", "hf", "synthetic"], "local": ["local"],
            "hf": ["hf"], "synthetic": ["synthetic"]}[DATA_SOURCE]

for src in _sources:
    try:
        if src == "local":
            FILES = _local_files()
        elif src == "hf":
            print("downloading from HuggingFace (needs Internet = ON) ...", flush=True)
            FILES = _hf_files()
        else:
            FILES = [("synthetic", i) for i in range(_need)]
            IS_SYNTHETIC = True
        if len(FILES) >= _need:
            DATA_ORIGIN = src; break
        print(f"  [{src}] only {len(FILES)} shapes found (need {_need}) -> next source")
        FILES = []
    except Exception as e:
        print(f"  [{src}] failed: {type(e).__name__}: {str(e)[:180]}")
        FILES = []

assert DATA_ORIGIN is not None, (
    f"No data source produced {_need} shapes. Turn Kaggle Internet ON, or set "
    f"DATA_SOURCE='synthetic' for a smoke test, or point LOCAL_DATA_ROOT at a PCC_DATA_ROOT.")
assert len(FILES) >= _need

print(f"\nsource   : {DATA_ORIGIN}")
print(f"category : {CATEGORY} ({SYNSET})")
print(f"shapes   : {len(FILES)} available, {_need} will be used")
if IS_SYNTHETIC:
    print("\n" + "!" * 78)
    print("!! SYNTHETIC FALLBACK IS ACTIVE — this is a SMOKE TEST, not a scientific result.")
    print("!! Any conclusion drawn from this run is about toy boxes, not ShapeNet objects.")
    print("!" * 78)

## 4 · Imports and reproducibility

Seeds for `random`, `numpy` and `torch`. `torch.use_deterministic_algorithms(warn_only=True)` asks
CUDA for deterministic kernels but never hard-fails when one is unavailable — **perfect GPU
determinism is not guaranteed**, and the notebook does not claim it. What *is* guaranteed, and is what
the fairness argument actually rests on, is that the paired models receive byte-identical data,
timesteps and noise (§13, §19), because those are drawn from explicitly seeded generators rather than
from ambient RNG state.

In [ ]:
import math, json, time, random, warnings
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3-D projection)
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 140, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25})


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
if DETERMINISTIC_STRICT:
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as e:
        print("deterministic algorithms not fully available:", type(e).__name__)

DEV = torch.device(DEVICE)
print("seeded with", SEED, "| device", DEV)
print("note: CUDA determinism is best-effort; paired fairness comes from explicit generators, not "
      "from global RNG state.")

## 5 · Dataset inspection

**Inspect:** that `xyz` is *not* yet centred/unit-scaled (it is raw ShapeNet-ish scale — §6 fixes
that), that `rgb` is already in `[0,1]`, and the count of distinct colours. A category whose objects
are nearly single-coloured has very little for geometry to predict, which is a real confound to keep
in mind when reading §24.

In [ ]:
def load_raw(entry):
    if IS_SYNTHETIC:
        return _synthetic_shape(seed=1000 + entry[1])
    z = np.load(entry)
    return (z["xyz"].astype(np.float32),
            np.clip(z["rgb"].astype(np.float32), 0.0, 1.0),
            z["part"].astype(np.int16))


_xyz, _rgb, _part = load_raw(FILES[0])
name0 = "synthetic-0" if IS_SYNTHETIC else os.path.basename(FILES[0])
print("example shape:", name0)
for nm, a in (("xyz", _xyz), ("rgb", _rgb), ("part", _part)):
    print(f"  {nm:5s} {str(a.shape):12s} {str(a.dtype):9s} "
          f"min {a.min():8.4f}  max {a.max():8.4f}")
print(f"  raw xyz centroid {np.round(_xyz.mean(0), 4)}  max radius "
      f"{np.linalg.norm(_xyz - _xyz.mean(0), axis=1).max():.4f}   -> NOT normalised yet")
print(f"  parts present    {sorted(np.unique(_part).tolist())}   "
      "(loaded for reporting only — NEVER fed to a model)")

_uc = np.unique((_rgb * 255).astype(np.uint8), axis=0)
print(f"  distinct colours {len(_uc)} / {len(_rgb)} points   rgb std {np.round(_rgb.std(0), 4)}")

_stats = []
for e in FILES[:min(30, len(FILES))]:
    x, c, _ = load_raw(e)
    _stats.append([len(np.unique((c * 255).astype(np.uint8), axis=0)), float(c.std())])
_stats = np.array(_stats)
print(f"\nover {len(_stats)} shapes: median distinct colours {np.median(_stats[:, 0]):.0f}, "
      f"median rgb std {np.median(_stats[:, 1]):.3f}")
print("  (low colour diversity => less for geometry to predict; noted as a confound in §24)")

## 6 · Preprocessing and normalisation

**The repository's existing procedure**, taken verbatim from `scripts/run_benchmark_kaggle.py`
(`make_entry` / `load_ours`):

$$X \leftarrow \frac{X-\bar X}{\max_i\|X_i-\bar X\|}\qquad\text{(centroid at origin, unit sphere)}$$

and RGB kept in $[0,1]$. Subsampling to `NUM_POINTS` happens **before** normalisation so the unit
sphere is exact for the points actually used — which is what makes the $[-1,1]$ clamp on the
reconstructed $\hat x_0$ (§8) provably valid rather than merely convenient.

Diffusion space is then

$$G_0 = X_{\text{unit}} \in[-1,1]^3, \qquad C_0 = 2\,\frac{RGB}{255}-1 = 2\,RGB_{[0,1]}-1 \in[-1,1]^3 .$$

Raw 0–255 RGB is never diffused next to normalised XYZ.

**Inspect:** the printed per-channel min / max / mean / std. Geometry std (~0.3) and colour std (~0.5)
are the same *order*, not identical — that is fine and expected, because we only ever compare **within**
a pair, where both models see the identical signal. It does mean the geometry and colour experiments
sit at different effective SNRs at the same $t$, so *the two experiments' absolute MSE values are not
comparable to each other* — only each pair's internal gap is. The SNR table below makes that explicit.

In [ ]:
# --- from scripts/run_benchmark_kaggle.py: make_entry() -------------------------------------
def normalize_xyz(xyz):
    c = xyz - xyz.mean(0)
    return (c / (np.linalg.norm(c, axis=1).max() + 1e-9)).astype(np.float32)


def fps_idx(xyz, n, start=0):
    '''Farthest-point sampling, deterministic (fixed start index). torch, CPU-friendly.'''
    pts = torch.from_numpy(np.ascontiguousarray(xyz)).float()
    N = pts.shape[0]
    idx = torch.zeros(n, dtype=torch.long)
    dist = torch.full((N,), 1e10)
    far = torch.tensor(start)
    for i in range(n):
        idx[i] = far
        dist = torch.minimum(dist, ((pts - pts[far]) ** 2).sum(-1))
        far = torch.max(dist, 0).indices
    return idx.numpy()


def subsample(xyz, rgb, n, seed):
    if len(xyz) == n:
        return xyz, rgb
    if SUBSAMPLE == "fps":
        idx = fps_idx(xyz, n)
    else:
        idx = np.random.default_rng(seed).choice(len(xyz), n, replace=len(xyz) < n)
    return xyz[idx], rgb[idx]


_SPLIT_OFFSET = {"train": 0, "val": 500000}   # fixed, NOT hash() — python string hashing is salted


def build_split(entries, tag):
    XYZ, RGB, IDS = [], [], []
    for i, e in enumerate(entries):
        xyz, rgb, _part = load_raw(e)                      # part deliberately dropped here
        xyz, rgb = subsample(xyz, rgb, NUM_POINTS, seed=SEED * 100003 + _SPLIT_OFFSET[tag] + i)
        XYZ.append(normalize_xyz(xyz)); RGB.append(np.clip(rgb, 0, 1).astype(np.float32))
        IDS.append("synthetic-%d" % e[1] if IS_SYNTHETIC else
                   os.path.splitext(os.path.basename(e))[0])
    return dict(xyz=np.stack(XYZ), rgb=np.stack(RGB), ids=IDS)


# ---- strictly disjoint object-level split -------------------------------------------------
_perm = np.random.default_rng(SEED).permutation(len(FILES))
_tr_i = _perm[:NUM_TRAIN_SHAPES]
_va_i = _perm[NUM_TRAIN_SHAPES:NUM_TRAIN_SHAPES + NUM_VAL_SHAPES]
assert len(set(_tr_i.tolist()) & set(_va_i.tolist())) == 0, "train/val object overlap!"

t0 = time.time()
DATA = {"train": build_split([FILES[i] for i in _tr_i], "train"),
        "val":   build_split([FILES[i] for i in _va_i], "val")}
assert len(set(DATA["train"]["ids"]) & set(DATA["val"]["ids"])) == 0, "train/val id overlap!"
print(f"built in {time.time()-t0:.1f}s  |  train {DATA['train']['xyz'].shape}  "
      f"val {DATA['val']['xyz'].shape}")

# ---- diffusion-space tensors --------------------------------------------------------------
for sp in ("train", "val"):
    DATA[sp]["G0"] = torch.from_numpy(DATA[sp]["xyz"]).float()             # unit sphere
    DATA[sp]["C0"] = torch.from_numpy(DATA[sp]["rgb"]).float() * 2.0 - 1.0  # [0,1] -> [-1,1]

print("\nfirst 3 train ids:", DATA["train"]["ids"][:3])
print("first 3 val   ids:", DATA["val"]["ids"][:3])

In [ ]:
# ---------------- preprocessing diagnostic ----------------
def describe(a, name):
    a = np.asarray(a).reshape(-1, 3)
    return pd.DataFrame({"channel": list("xyz") if name.startswith(("XYZ", "G")) else list("rgb"),
                         "min": a.min(0), "max": a.max(0), "mean": a.mean(0), "std": a.std(0)}
                        ).assign(tensor=name)


_rows = []
for sp in ("train", "val"):
    _rows.append(describe(DATA[sp]["xyz"], f"XYZ unit-sphere [{sp}]"))
    _rows.append(describe(DATA[sp]["rgb"], f"RGB [0,1] [{sp}]"))
    _rows.append(describe(DATA[sp]["G0"].numpy(), f"G0 = XYZ [{sp}]"))
    _rows.append(describe(DATA[sp]["C0"].numpy(), f"C0 = 2*RGB-1 [{sp}]"))
NORM_STATS = pd.concat(_rows)[["tensor", "channel", "min", "max", "mean", "std"]].reset_index(drop=True)
NORM_STATS.to_csv(os.path.join(TAB_DIR, "normalization_stats.csv"), index=False)
print(NORM_STATS.to_string(index=False, float_format=lambda v: f"{v: .4f}"))

_r = np.linalg.norm(DATA["train"]["xyz"], axis=-1)
print(f"\nmax radius over all train points: {_r.max():.6f}  (must be <= 1.0 -> the [-1,1] clamp in "
      f"§8 is valid)")
assert _r.max() <= 1.0 + 1e-5
assert DATA["train"]["C0"].abs().max() <= 1.0 + 1e-6

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
for c, lab in zip(range(3), "xyz"):
    axes[0].hist(DATA["train"]["G0"][..., c].numpy().ravel(), bins=80, alpha=.55, label=lab)
for c, lab in zip(range(3), ["r", "g", "b"]):
    axes[1].hist(DATA["train"]["C0"][..., c].numpy().ravel(), bins=80, alpha=.55, label=lab)
axes[0].set_title("$G_0$ (XYZ, unit sphere)"); axes[1].set_title("$C_0 = 2\\,RGB-1$")
for a in axes: a.legend(fontsize=7); a.set_xlim(-1.05, 1.05)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "06_normalization.png")); plt.show(); plt.close(fig)

## 7 · Visualisation sanity check

Three training objects rendered with their true colours, using the exact camera
(`VIEW_ELEV`, `VIEW_AZIM`) that every later 3-D panel reuses.

**Inspect:** they should look like recognisable objects with sensible colours. If they look like a
formless blob, the normalisation or the axis convention is wrong and nothing downstream is meaningful.

In [ ]:
def plot_cloud(ax, xyz, rgb, title="", lim=1.05, s=None, cmap_vals=None, cmap="magma",
               vmin=None, vmax=None):
    '''One 3-D panel. ShapeNet is Y-up, so we plot (x, z, y) to stand the object upright.
    Every panel in this notebook uses the same camera and the same axis limits.'''
    s = PT_SIZE if s is None else s
    if cmap_vals is None:
        sc = ax.scatter(xyz[:, 0], xyz[:, 2], xyz[:, 1], c=np.clip(rgb, 0, 1),
                        s=s, linewidths=0, depthshade=False)
    else:
        sc = ax.scatter(xyz[:, 0], xyz[:, 2], xyz[:, 1], c=cmap_vals, cmap=cmap,
                        vmin=vmin, vmax=vmax, s=s, linewidths=0, depthshade=False)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    try:
        ax.set_box_aspect((1, 1, 1))
    except Exception:
        pass
    ax.view_init(VIEW_ELEV, VIEW_AZIM); ax.set_axis_off()
    ax.set_title(title, fontsize=8)
    return sc


def grid3d(panels, ncols=4, figsize_per=2.5, suptitle=None, path=None, **kw):
    n = len(panels); nrows = int(np.ceil(n / ncols))
    fig = plt.figure(figsize=(figsize_per * ncols, figsize_per * nrows + 0.4))
    for i, (title, xyz, rgb) in enumerate(panels):
        ax = fig.add_subplot(nrows, ncols, i + 1, projection="3d")
        plot_cloud(ax, xyz, rgb, title, **kw)
    if suptitle: fig.suptitle(suptitle, fontsize=10)
    fig.tight_layout()
    if path: fig.savefig(path, bbox_inches="tight")
    plt.show(); plt.close(fig)
    return fig


grid3d([(f"train[{i}]  {DATA['train']['ids'][i][:14]}", DATA["train"]["xyz"][i],
         DATA["train"]["rgb"][i]) for i in range(3)],
       ncols=3, suptitle=f"{CATEGORY} — normalised training clouds ({NUM_POINTS} pts)",
       path=os.path.join(FIG_DIR, "07_dataset_check.png"))

## 8 · DDPM forward process

$$\alpha_t = 1-\beta_t,\qquad \bar\alpha_t=\prod_{s=1}^{t}\alpha_s,\qquad
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\ \ \epsilon\sim\mathcal N(0,I),$$

with `T = 1000` and the cosine schedule (Nichol & Dhariwal), reusing `cosine_betas` from
`scripts/run_benchmark_kaggle.py` — the repo already uses it for low-dimensional point signals.
Training draws $t\sim\mathrm{Uniform}\{0,\dots,T-1\}$.

Experiment 1 noises **geometry only** ($C_0$ stays clean); Experiment 2 noises **colour only**
($G_0$ stays clean).

Clean-signal reconstruction:

$$\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\,\hat\epsilon}{\sqrt{\bar\alpha_t}},$$

clamped to $[-1,1]$ — valid for both modalities because §6 guarantees $|G_0|\le1$ and
$C_0\in[-1,1]$. The clamp matters: at $t=900$, $\sqrt{\bar\alpha_t}$ is small enough that the
division amplifies any $\hat\epsilon$ error by a large factor, so the *unclamped* $\hat x_0$ is
meaningless. It is applied identically to both models of a pair.

**Inspect:** the four assertions below are pure numerics — they catch a broken schedule or a wrong
reconstruction formula *without training anything*. If any fails, stop and fix before §12.

In [ ]:
def make_betas(T, schedule):
    if schedule == "cosine":
        # scripts/run_benchmark_kaggle.py :: cosine_betas  (Nichol & Dhariwal, s=0.008)
        s = 0.008
        t = torch.linspace(0, T, T + 1) / T
        f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
        ab = f / f[0]
        return (1 - ab[1:] / ab[:-1]).clamp(1e-8, 0.999)
    if schedule == "linear":
        return torch.linspace(1e-4, 0.02, T)
    raise ValueError(schedule)


class Diffusion:
    '''Minimal DDPM forward process (no reverse sampler — this study never samples).'''

    def __init__(self, T, schedule, device):
        b = make_betas(T, schedule).to(device)
        a = 1.0 - b
        abar = torch.cumprod(a, 0)
        self.T, self.device = T, device
        self.betas, self.alphas, self.abar = b, a, abar
        self.sqrt_abar = abar.sqrt()
        self.sqrt_1mabar = (1 - abar).sqrt()

    @staticmethod
    def _ext(v, t, ndim):
        return v[t].view(-1, *([1] * (ndim - 1)))

    def q_sample(self, x0, t, noise):
        return (self._ext(self.sqrt_abar, t, x0.dim()) * x0
                + self._ext(self.sqrt_1mabar, t, x0.dim()) * noise)

    def x0_from_eps(self, x_t, t, eps, clamp=(-1.0, 1.0)):
        sa = self._ext(self.sqrt_abar, t, x_t.dim()).clamp(min=1e-5)
        sb = self._ext(self.sqrt_1mabar, t, x_t.dim())
        x0 = (x_t - sb * eps) / sa
        return x0.clamp(*clamp) if clamp is not None else x0


DIF = Diffusion(T, BETA_SCHEDULE, DEV)

# ---- assertions on the forward process (no learning involved) ----
ab = DIF.abar
assert torch.all(ab[1:] <= ab[:-1] + 1e-9), "abar must be non-increasing"
assert ab[0] > 0.99, f"abar[0] = {ab[0]:.5f} — the first step destroys too much signal"
print(f"abar[0]={ab[0]:.6f}  abar[T//2]={ab[T//2]:.6f}  abar[T-1]={ab[-1]:.3e}")

_x0 = DATA["val"]["G0"][:4].to(DEV)
_t0 = torch.zeros(4, dtype=torch.long, device=DEV)
_n = torch.randn_like(_x0)
_err0 = (DIF.q_sample(_x0, _t0, _n) - _x0).abs().max().item()
print(f"q_sample at t=0 deviates from x0 by at most {_err0:.4f}  (should be small)")
assert _err0 < 0.2

print("\nround-trip  x0 -> q_sample -> x0_from_eps(true eps)   [unclamped, float32]")
_ok = True
for tt in [0, 50, 100, 250, 500, 750, 900, 990]:
    t = torch.full((4,), tt, dtype=torch.long, device=DEV)
    xt = DIF.q_sample(_x0, t, _n)
    rec = DIF.x0_from_eps(xt, t, _n, clamp=None)
    e = (rec - _x0).abs().max().item()
    _ok &= e < 1e-2
    print(f"   t={tt:4d}  sqrt(abar)={DIF.sqrt_abar[tt]:.5f}   max|err| = {e:.2e}")
assert _ok, "clean-signal reconstruction formula is wrong"
print("\nDDPM forward process: all checks passed.")

In [ ]:
# ---- schedule + per-modality SNR diagnostic ----
sd_g = float(DATA["train"]["G0"].std())
sd_c = float(DATA["train"]["C0"].std())
snr = pd.DataFrame({"t": EVAL_TIMESTEPS})
snr["sqrt_abar"] = [float(DIF.sqrt_abar[t]) for t in EVAL_TIMESTEPS]
snr["sqrt_1-abar"] = [float(DIF.sqrt_1mabar[t]) for t in EVAL_TIMESTEPS]
snr["SNR_geometry"] = snr["sqrt_abar"] * sd_g / snr["sqrt_1-abar"]
snr["SNR_colour"] = snr["sqrt_abar"] * sd_c / snr["sqrt_1-abar"]
snr.to_csv(os.path.join(TAB_DIR, "snr_by_timestep.csv"), index=False)
print(f"signal std: geometry {sd_g:.3f}   colour {sd_c:.3f}")
print("SNR_modality = sqrt(abar)*std / sqrt(1-abar)  -> the SAME t is a HARDER corruption for the "
      "lower-std modality,\nso absolute MSE is comparable WITHIN a pair only, never across the two "
      "experiments.\n")
print(snr.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

fig, axes = plt.subplots(1, 3, figsize=(10, 2.6))
axes[0].plot(DIF.betas.cpu()); axes[0].set_title(r"$\beta_t$"); axes[0].set_xlabel("t")
axes[1].plot(DIF.abar.cpu()); axes[1].set_title(r"$\bar\alpha_t$"); axes[1].set_xlabel("t")
axes[2].plot(DIF.sqrt_abar.cpu(), label=r"$\sqrt{\bar\alpha_t}$")
axes[2].plot(DIF.sqrt_1mabar.cpu(), label=r"$\sqrt{1-\bar\alpha_t}$")
for t in EVAL_TIMESTEPS: axes[2].axvline(t, color="k", lw=.4, alpha=.4)
axes[2].legend(fontsize=7); axes[2].set_title("mixing coefficients (| = evaluated t)")
fig.suptitle(f"{BETA_SCHEDULE} schedule, T={T}", fontsize=10)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "08_schedule.png")); plt.show(); plt.close(fig)

In [ ]:
# ---- what the corruption actually looks like, for both modalities ----
_i = 0
_g0 = DATA["val"]["G0"][_i:_i + 1].to(DEV); _c0 = DATA["val"]["C0"][_i:_i + 1].to(DEV)
_gen = torch.Generator(device="cpu").manual_seed(SEED)
_nz = torch.randn(1, NUM_POINTS, 3, generator=_gen).to(DEV)
rgb0 = DATA["val"]["rgb"][_i]

pan_g, pan_c = [("t=0 (clean)", DATA["val"]["xyz"][_i], rgb0)], [("t=0 (clean)", DATA["val"]["xyz"][_i], rgb0)]
for tt in EVAL_TIMESTEPS:
    t = torch.full((1,), tt, dtype=torch.long, device=DEV)
    gt_ = DIF.q_sample(_g0, t, _nz)[0].cpu().numpy()
    ct_ = ((DIF.q_sample(_c0, t, _nz)[0].cpu().numpy() + 1) / 2).clip(0, 1)
    pan_g.append((f"$G_t$, t={tt}", gt_, rgb0))
    pan_c.append((f"$C_t$, t={tt}", DATA["val"]["xyz"][_i], ct_))

grid3d(pan_g, ncols=7, figsize_per=2.0, lim=1.6,
       suptitle="Experiment 1 — geometry is noised, colour stays clean (axes clipped to ±1.6)",
       path=os.path.join(FIG_DIR, "08_forward_geometry.png"))
grid3d(pan_c, ncols=7, figsize_per=2.0,
       suptitle="Experiment 2 — colour is noised, geometry stays clean (colours clipped to [0,1] "
                "for display only)",
       path=os.path.join(FIG_DIR, "08_forward_colour.png"))

## 9 · Timestep embedding

Standard sinusoidal embedding → 2-layer MLP → FiLM (per-channel scale & shift) injected into **every**
block of **all four** models. Identical mechanism, identical width, identical injection point
everywhere: the timestep pathway is never part of what differs between a baseline and its conditioned
counterpart.

Taken from `scripts/run_benchmark_kaggle.py :: timestep_embedding`.

In [ ]:
def timestep_embedding(t, dim):
    '''(B,) int timesteps -> (B, dim) sinusoidal features.'''
    half = dim // 2
    f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    a = t.float().view(-1, 1) * f.view(1, -1)
    return torch.cat([a.sin(), a.cos()], -1)


_e = timestep_embedding(torch.arange(0, T, 10, device=DEV), WIDTH).cpu().numpy()
assert _e.shape == (T // 10, WIDTH)
assert np.abs(_e[0] - _e[-1]).max() > 0.5, "embedding does not separate t=0 from t=T"
fig, ax = plt.subplots(figsize=(6, 2.4))
im = ax.imshow(_e.T, aspect="auto", cmap="twilight", origin="lower",
               extent=[0, T, 0, WIDTH])
ax.set_xlabel("t"); ax.set_ylabel("embedding dim"); ax.set_title("sinusoidal timestep embedding")
ax.grid(False); fig.colorbar(im, ax=ax, fraction=.03)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "09_timestep_embedding.png")); plt.show(); plt.close(fig)
print("distinct-t separation ok |", _e.shape)

## 10 · Model definitions

One class, `PointDenoiser`, instantiated four times. Structure per point $i$:

```
input  [B, N, 3 + cond_dim]           ->  shared Linear
  x N_BLOCKS:   h <- h + fuse([ h , maxpool_over_points(h) (, EdgeConv(h) ) ]) * (1+scale(t)) + shift(t)
output [B, N, 3]                       =  predicted noise  eps_hat
```

* **shared per-point MLP** + **permutation-invariant global max-pool** (the classic PointNet global
  feature) + **FiLM by the timestep embedding**. Point cardinality is preserved: `[B,N,D] -> [B,N,3]`.
* The block layout is lifted from the repo's `Block` in `scripts/run_benchmark_kaggle.py`, minus the
  part-pooling branch (no part labels here) and with the EdgeConv branch off by default.

| model | input | conditioned on | target | `cond_dim` |
|---|---|---|---|---|
| **G**   | $G_t$ | — | $\epsilon_g$ | 0 |
| **G+C** | $[G_t, C_0]$ | clean colour | $\epsilon_g$ | 3 |
| **C**   | $C_t$ | — | $\epsilon_c$ | 0 |
| **C+G** | $[C_t, G_0]$ | clean geometry | $\epsilon_c$ | 3 |

A pair differs by exactly one `Linear(3→WIDTH)` worth of weights — ≈0.1 % of the parameter count,
printed below. Nobody is comparing a small baseline to a big conditioned model.

### The geometry/colour design rule, enforced

The default backbone is **strictly pointwise**: no kNN, no FPS, no ball query, no graph. It therefore
cannot express a mixed XYZ+RGB distance even by accident, and `Model C` is *structurally* geometry-blind.

`USE_LOCAL_GEOMETRY_BACKBONE=True` adds an EdgeConv branch **for the geometry pair only**, whose kNN
graph is built from `coords = G_t` — a tensor **both** `Model G` and `Model G+C` hold, so the pair stays
fair and colour still never touches a neighbourhood. It is refused for the colour pair, because there
the only available coordinates are $G_0$, which `Model C` must not see: enabling it would either leak
geometry into the baseline or make the pair unfair. The assertion below encodes that.

**Inspect:** the leakage table. Each model is fed a *perturbed* version of the modality it is supposed
to be blind to; a blind model's output must change by **exactly 0.0**.

In [ ]:
def _gather_nb(h, idx):                       # (B,N,C),(B,N,k) -> (B,N,k,C)
    B, N, C = h.shape
    off = (torch.arange(B, device=h.device) * N).view(B, 1, 1)
    return h.reshape(B * N, C)[(idx + off).reshape(-1)].reshape(B, N, idx.shape[-1], C)


def knn_graph(xyz, k, chunk=2048):
    '''(B,N,3) XYZ -> (B,N,k) neighbour indices, self excluded.
    XYZ ONLY — this function is never called with anything but spatial coordinates.'''
    assert xyz.shape[-1] == 3, "neighbourhoods must be built from 3-D spatial coordinates only"
    B, N, _ = xyz.shape
    out = torch.empty(B, N, k, dtype=torch.long, device=xyz.device)
    kk = min(k + 1, N)
    for s in range(0, N, chunk):
        d = torch.cdist(xyz[:, s:s + chunk], xyz)
        idx = d.topk(kk, dim=-1, largest=False).indices[:, :, 1:]
        if idx.shape[-1] < k:
            idx = idx[..., [i % idx.shape[-1] for i in range(k)]]
        out[:, s:s + chunk] = idx
    return out


class Block(nn.Module):
    '''Global-context residual block, FiLM-modulated by t. Optional XYZ-only EdgeConv branch.'''

    def __init__(self, w, use_local=False):
        super().__init__()
        self.use_local = use_local
        if use_local:
            self.edge = nn.Sequential(nn.Linear(2 * w + 4, w), nn.GELU(), nn.Linear(w, w))
        ctx = w * (3 if use_local else 2)
        self.fuse = nn.Sequential(nn.LayerNorm(ctx), nn.Linear(ctx, w), nn.GELU(), nn.Linear(w, w))
        self.film = nn.Linear(w, 2 * w)

    def forward(self, h, temb, ctx=None):
        g = h.max(1, keepdim=True).values.expand_as(h)        # global feature, permutation-invariant
        feats = [h, g]
        if self.use_local:
            hj = _gather_nb(h, ctx["idx"]); hi = h.unsqueeze(2).expand_as(hj)
            feats.append(self.edge(torch.cat([hi, hj - hi, ctx["rel"]], -1)).max(2).values)
        d = self.fuse(torch.cat(feats, -1))
        sc, sh = self.film(temb).unsqueeze(1).chunk(2, -1)
        return h + d * (1 + sc) + sh


class PointDenoiser(nn.Module):
    '''eps-prediction on an (N,3) per-point signal. Same class for all four configurations.'''

    def __init__(self, cond_dim=0, width=128, n_blocks=3, use_local=False, k=16):
        super().__init__()
        self.cond_dim, self.width, self.use_local, self.k = cond_dim, width, use_local, k
        self.inp = nn.Linear(3 + cond_dim, width)
        self.temb = nn.Sequential(nn.Linear(width, width), nn.SiLU(), nn.Linear(width, width))
        self.blocks = nn.ModuleList([Block(width, use_local) for _ in range(n_blocks)])
        self.out = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.GELU(),
                                 nn.Linear(width, 3))

    def build_ctx(self, coords):
        idx = knn_graph(coords, self.k)                       # XYZ only (asserted inside)
        rel = _gather_nb(coords, idx) - coords.unsqueeze(2)
        scale = rel.norm(dim=-1).mean(dim=(1, 2), keepdim=True).clamp(min=1e-6).unsqueeze(-1)
        return dict(idx=idx, rel=torch.cat([rel / scale, rel.norm(dim=-1, keepdim=True) / scale], -1))

    def forward(self, x_t, t, cond=None, coords=None):
        assert (cond is None) == (self.cond_dim == 0), "cond presence must match cond_dim"
        f = x_t if cond is None else torch.cat([x_t, cond], -1)
        h = self.inp(f)
        if t.dim() == 0:
            t = t.expand(x_t.shape[0])
        temb = self.temb(timestep_embedding(t, self.width))
        ctx = self.build_ctx(coords) if self.use_local else None
        if self.use_local:
            assert coords is not None and coords.shape[-1] == 3
        for b in self.blocks:
            h = b(h, temb, ctx)
        return self.out(h)


MODEL_SPECS = {
    "G":   dict(pair="geometry", target="G0", cond=None, blind_to="C0",
                label="Model G — geometry-only",              short="XYZ only"),
    "G+C": dict(pair="geometry", target="G0", cond="C0", blind_to=None,
                label="Model G+C — geometry | clean colour",   short="XYZ + RGB"),
    "C":   dict(pair="colour",   target="C0", cond=None, blind_to="G0",
                label="Model C — colour-only",                 short="RGB only"),
    "C+G": dict(pair="colour",   target="C0", cond="G0", blind_to=None,
                label="Model C+G — colour | clean geometry",    short="RGB + XYZ"),
}
PAIRS = {"geometry": ("G", "G+C"), "colour": ("C", "C+G")}


def uses_local(key):
    return bool(USE_LOCAL_GEOMETRY_BACKBONE) and MODEL_SPECS[key]["pair"] == "geometry"


assert not any(uses_local(k) for k in ("C", "C+G")), \
    "the local backbone must never be enabled for the colour pair (it would leak geometry into Model C)"


def build_model(key):
    spec = MODEL_SPECS[key]
    set_seed(SEED)                      # every model starts from the same RNG state
    return PointDenoiser(cond_dim=0 if spec["cond"] is None else 3, width=WIDTH,
                         n_blocks=N_BLOCKS, use_local=uses_local(key), k=LOCAL_K)


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
# ---------------- parameter counts + shape smoke test ----------------
_probe = {k: build_model(k).to(DEV) for k in MODEL_SPECS}
_B = 2
_g0 = DATA["val"]["G0"][:_B].to(DEV); _c0 = DATA["val"]["C0"][:_B].to(DEV)
_t = torch.full((_B,), 300, dtype=torch.long, device=DEV)


def _inputs(key, g_t=None, c_t=None, g0=None, c0=None):
    '''Assemble the full call signature (x_t, t, cond, coords) for a model.
    `coords` is XYZ only, and only ever populated for the optional local backbone.'''
    spec = MODEL_SPECS[key]
    g0 = _g0 if g0 is None else g0; c0 = _c0 if c0 is None else c0
    x_t = (g_t if g_t is not None else g0) if spec["target"] == "G0" else (c_t if c_t is not None else c0)
    cond = None if spec["cond"] is None else (c0 if spec["cond"] == "C0" else g0)
    coords = x_t if uses_local(key) else None     # G_t: available to BOTH models of the geometry pair
    return x_t, _t, cond, coords


rows = []
for k, m in _probe.items():
    with torch.no_grad():
        o = m(*_inputs(k))
    assert o.shape == (_B, NUM_POINTS, 3), f"{k}: expected (B,N,3), got {tuple(o.shape)}"
    rows.append(dict(model=k, description=MODEL_SPECS[k]["label"], input=MODEL_SPECS[k]["short"],
                     in_channels=3 + (0 if MODEL_SPECS[k]["cond"] is None else 3),
                     local_backbone=uses_local(k), params=n_params(m), out_shape=str(tuple(o.shape))))
PARAM_TABLE = pd.DataFrame(rows)
PARAM_TABLE.to_csv(os.path.join(TAB_DIR, "model_parameters.csv"), index=False)
print(PARAM_TABLE.to_string(index=False))
for p, (b, c) in PAIRS.items():
    pb = PARAM_TABLE.set_index("model").loc[b, "params"]
    pc = PARAM_TABLE.set_index("model").loc[c, "params"]
    print(f"\n{p:9s} pair: {b} {pb:,} vs {c} {pc:,} params  -> "
          f"{100*(pc-pb)/pb:+.2f}% (conditioning adds one Linear(3->{WIDTH}))")

In [ ]:
# ---------------- EXECUTABLE LEAKAGE TEST ----------------
# The corrupted input x_t is held FIXED while each auxiliary tensor is perturbed in turn. A model
# that cannot see a modality must return a BIT-IDENTICAL output (max|delta| exactly 0.0); a model
# that is conditioned on it must move. This is a property of the wiring, so it holds before and
# after training — §18 re-runs the very same function on the trained weights.
def leakage_table(models, n=4, t_val=300):
    g0 = DATA["val"]["G0"][:n].to(DEV); c0 = DATA["val"]["C0"][:n].to(DEV)
    tv = torch.full((n,), t_val, dtype=torch.long, device=DEV)
    gen = torch.Generator().manual_seed(11)
    nz = torch.randn(n, NUM_POINTS, 3, generator=gen).to(DEV)
    xt = {"G0": DIF.q_sample(g0, tv, nz), "C0": DIF.q_sample(c0, tv, nz)}
    R = torch.linalg.qr(torch.randn(3, 3, generator=torch.Generator().manual_seed(7)))[0].to(DEV)
    g0_alt = g0 @ R                                # a rotated, equally valid geometry
    c0_alt = torch.tanh(c0 * 2.3 + 0.4)            # different, still-valid colours in [-1,1]

    rows = []
    for k, m in models.items():
        spec = MODEL_SPECS[k]
        x_t = xt[spec["target"]]
        coords = x_t if uses_local(k) else None    # XYZ only, and only for the geometry pair

        def out(gg, cc):
            cond = None if spec["cond"] is None else (cc if spec["cond"] == "C0" else gg)
            with torch.no_grad():
                return m(x_t, tv, cond, coords)

        base = out(g0, c0)
        d_g = (out(g0_alt, c0) - base).abs().max().item()
        d_c = (out(g0, c0_alt) - base).abs().max().item()
        exp_g = "uses" if spec["cond"] == "G0" else "blind"
        exp_c = "uses" if spec["cond"] == "C0" else "blind"
        ok = ((d_g > 0) if exp_g == "uses" else (d_g == 0.0)) and \
             ((d_c > 0) if exp_c == "uses" else (d_c == 0.0))
        rows.append(dict(model=k, expect_geometry=exp_g, expect_colour=exp_c,
                         d_out_perturb_G0=d_g, d_out_perturb_C0=d_c,
                         verdict="PASS" if ok else "FAIL"))
    return pd.DataFrame(rows)


print("leakage test — x_t held fixed, each auxiliary modality perturbed in turn:\n")
LEAK_TABLE = leakage_table(_probe)
LEAK_TABLE.to_csv(os.path.join(TAB_DIR, "leakage_test.csv"), index=False)
print(LEAK_TABLE.to_string(index=False))
assert (LEAK_TABLE.verdict == "PASS").all(), "modality leakage detected — do not trust any result"
print("\nAll four models see exactly the modalities their definition allows:")
print("  Model G   ignores colour entirely (both deltas 0 — it has no conditioning input at all).")
print("  Model C   ignores geometry entirely — this is the one that matters most, and it holds")
print("            structurally, because the backbone builds no neighbourhood of any kind.")
print("  Model G+C moves only when colour changes; Model C+G moves only when geometry changes.")
del _probe

## 11 · Metrics

Every metric is defined mathematically here and implemented once, so a baseline and its conditioned
counterpart are scored by literally the same code on literally the same clouds.

### Geometry (all ↓ lower is better)

| metric | definition | what it catches |
|---|---|---|
| `eps_mse` | $\frac1{3N}\|\epsilon_g-\hat\epsilon_g\|_2^2$ | the training objective itself |
| `x0_mse` | $\frac1{3N}\|G_0-\hat G_0\|_2^2$ | error in the *recovered clean shape* |
| `x0_p2p` | $\frac1N\sum_i\|G_{0,i}-\hat G_{0,i}\|_2$ | mean per-point displacement, in unit-sphere units |
| `chamfer_l1` | $\frac1N\sum_a\min_b\|a-b\| + \frac1N\sum_b\min_a\|a-b\|$ | shape-level fit ignoring correspondence |
| `chamfer_l2` | same with squared distances | as above, outlier-sensitive |
| `colored_chamfer_dE` | mean $\Delta E_{76}$ between each predicted point's (unchanged) colour and the colour of its nearest GT point | **did geometry error push points across colour boundaries** — the failure that actually hurts colored completion |

### Colour (for every model that reconstructs colour)

Pointwise, in $[0,1]$ RGB and in CIELAB:

| metric | definition | direction |
|---|---|---|
| `rgb_mae` | $\frac1{3N}\sum\lvert c-\hat c\rvert$ | ↓ |
| `rgb_mse` | $\frac1{3N}\sum(c-\hat c)^2$ | ↓ |
| `psnr` | $10\log_{10}\!\big(1/\texttt{rgb\_mse}\big)$, peak $=1$ | ↑ |
| `deltaE76` | $\lVert \mathrm{Lab}(c)-\mathrm{Lab}(\hat c)\rVert_2$, averaged | ↓ |
| `deltaE00` | CIEDE2000, averaged | ↓ |

$\Delta E_{76}$ is the repo's existing metric (`scripts/eval_fidelity.py`), kept for continuity;
$\Delta E_{00}$ is added because CIE76 over-weights saturated colours, and the two can disagree.

Beyond pointwise error — each of the following exists to catch a failure mode the pointwise numbers
are blind to:

**`local_consistency_absdiff` ↓ / `local_consistency_ratio` (→1).**
With $\mathcal N_k(i)$ the $k$ nearest neighbours **of point $i$ in XYZ** (never RGB), define the local
colour roughness

$$s(C)=\frac1{N}\sum_i \frac1{k}\sum_{j\in\mathcal N_k(i)} \lVert c_i-c_j\rVert_2 .$$

Report $\lvert s(\hat C)-s(C_0)\rvert$ and $s(\hat C)/s(C_0)$. *Detects:* salt-and-pepper colour noise
(ratio ≫ 1) and over-smoothing / colour blur (ratio ≪ 1). Both can coexist with a decent RGB MAE.

**`hist_chi2` ↓.** $\chi^2$ distance between $8^3$-bin RGB histograms,
$\tfrac12\sum_b (p_b-q_b)^2/(p_b+q_b)$. *Detects:* a globally wrong palette.

**`swd_lab` ↓.** Sliced Wasserstein-1 between the two Lab point sets over 64 fixed random directions
(sorted-projection form). *Detects:* distribution shift that binning misses; unlike $\chi^2$ it is
sensitive to *how far* the colours moved, not just to which bin they fell in.

Both distribution metrics are deliberately **blind to position** — that is what makes the next two
informative by contrast.

**`region_dE` ↓ (region-wise colour error) — the geometry-aware metric.** Partition the cloud with
`N_REGIONS` FPS seeds on $G_0$, assigning each point to its nearest seed: an **XYZ-only, colour-blind
Voronoi partition**, computed once per shape and shared by both models of the pair. With $\mu_m(\cdot)$
the mean colour of cell $m$,

$$\texttt{region\_dE}=\frac{\sum_m |m|\;\Delta E_{76}\big(\mathrm{Lab}(\mu_m(\hat C)),\,\mathrm{Lab}(\mu_m(C_0))\big)}{\sum_m |m|}.$$

*Detects:* **right palette in the wrong place.** A model that paints the seat's colour onto the legs
matches the RGB histogram exactly and fails here, because a spatially scrambled colour field drives
every cell mean toward the object's global mean. This is the geometry-aware stand-in for the part
labels, which are deliberately not used.

**`region_dE_norm` ↓ ∈ [0, ~1].** The same quantity divided by its **chance level** — `region_dE`
evaluated on a randomly permuted copy of the GT colours, i.e. the identical colour multiset stripped
of all spatial information. **0 = colours in exactly the right regions, 1 = no better than a
scrambled palette.** The normalisation cancels the object's own colour diversity, so the number is
comparable across objects in a way the raw ΔE is not. Reported as `NaN` (and excluded from averages)
when the chance level is itself below 1 ΔE unit, which means the object is too close to
single-coloured for the ratio to mean anything; §20 prints the chance level so you can see how near
that regime the category sits.

**`colour_edge_iou` ↑ ∈ [0,1].** Region means say nothing about *boundaries*. Define the local colour
contrast at point $i$ over its XYZ neighbourhood, $e_i(C)=\max_{j\in\mathcal N_k(i)}\lVert c_i-c_j\rVert$,
threshold at $\tau=\mathrm{P}_{90}\big(e(C_0)\big)$ — **the same $\tau$, derived from ground truth, for
both models** — and report the IoU of the two resulting edge-point sets:

$$\texttt{colour\_edge\_iou}=\frac{\big|\{e(\hat C)>\tau\}\cap\{e(C_0)>\tau\}\big|}
{\big|\{e(\hat C)>\tau\}\cup\{e(C_0)>\tau\}\big|}.$$

*Detects:* whether colour *transitions* land on the right geometry. An over-smoothed prediction has
no points above $\tau$ and scores 0; one that hallucinates texture puts edges in the wrong places and
also scores low; an exact reconstruction scores 1. This is the point-cloud analogue of edge-based
image-quality measures, and it is the one metric here that is sensitive to material boundaries rather
than to average colour.

In [ ]:
# ---- CIELAB, verbatim from scripts/eval_fidelity.py -------------------------------------
def srgb_to_lab(rgb):
    rgb = np.clip(np.asarray(rgb, float), 0, 1)
    lin = np.where(rgb > 0.04045, ((rgb + 0.055) / 1.055) ** 2.4, rgb / 12.92)
    M = np.array([[0.4124, 0.3576, 0.1805],
                  [0.2126, 0.7152, 0.0722],
                  [0.0193, 0.1192, 0.9505]])
    xyz = (lin @ M.T) / np.array([0.95047, 1.0, 1.08883])
    d = 6 / 29
    f = np.where(xyz > d ** 3, np.cbrt(xyz), xyz / (3 * d ** 2) + 4 / 29)
    return np.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)


def delta_e76(a, b):
    return np.linalg.norm(srgb_to_lab(a) - srgb_to_lab(b), axis=1)


def delta_e00_lab(lab1, lab2):
    '''CIEDE2000 between two (N,3) Lab arrays. kL=kC=kH=1.'''
    L1, a1, b1 = lab1[:, 0], lab1[:, 1], lab1[:, 2]
    L2, a2, b2 = lab2[:, 0], lab2[:, 1], lab2[:, 2]
    C1, C2 = np.hypot(a1, b1), np.hypot(a2, b2)
    Cbar = (C1 + C2) / 2.0
    G = 0.5 * (1 - np.sqrt(Cbar ** 7 / (Cbar ** 7 + 25.0 ** 7 + 1e-30)))
    a1p, a2p = (1 + G) * a1, (1 + G) * a2
    C1p, C2p = np.hypot(a1p, b1), np.hypot(a2p, b2)
    h1p = np.degrees(np.arctan2(b1, a1p)) % 360.0
    h2p = np.degrees(np.arctan2(b2, a2p)) % 360.0
    h1p = np.where(C1p == 0, 0.0, h1p); h2p = np.where(C2p == 0, 0.0, h2p)

    dLp = L2 - L1
    dCp = C2p - C1p
    dh = h2p - h1p
    dhp = np.where(C1p * C2p == 0, 0.0,
                   np.where(np.abs(dh) <= 180, dh, np.where(dh > 180, dh - 360, dh + 360)))
    dHp = 2 * np.sqrt(C1p * C2p) * np.sin(np.radians(dhp) / 2.0)

    Lbp = (L1 + L2) / 2.0
    Cbp = (C1p + C2p) / 2.0
    hsum, hdif = h1p + h2p, np.abs(h1p - h2p)
    hbp = np.where(C1p * C2p == 0, hsum,
                   np.where(hdif <= 180, hsum / 2.0,
                            np.where(hsum < 360, (hsum + 360) / 2.0, (hsum - 360) / 2.0)))

    Tt = (1 - 0.17 * np.cos(np.radians(hbp - 30)) + 0.24 * np.cos(np.radians(2 * hbp))
          + 0.32 * np.cos(np.radians(3 * hbp + 6)) - 0.20 * np.cos(np.radians(4 * hbp - 63)))
    dtheta = 30 * np.exp(-(((hbp - 275) / 25.0) ** 2))
    Rc = 2 * np.sqrt(Cbp ** 7 / (Cbp ** 7 + 25.0 ** 7 + 1e-30))
    SL = 1 + 0.015 * (Lbp - 50) ** 2 / np.sqrt(20 + (Lbp - 50) ** 2)
    SC = 1 + 0.045 * Cbp
    SH = 1 + 0.015 * Cbp * Tt
    RT = -np.sin(np.radians(2 * dtheta)) * Rc
    return np.sqrt((dLp / SL) ** 2 + (dCp / SC) ** 2 + (dHp / SH) ** 2
                   + RT * (dCp / SC) * (dHp / SH))


def delta_e00(a_rgb, b_rgb):
    return delta_e00_lab(srgb_to_lab(a_rgb), srgb_to_lab(b_rgb))

In [ ]:
# ---------------- CIEDE2000 self-test (Sharma et al. reference pairs) ----------------
# Soft check: if it fails, deltaE00 is still reported but deltaE76 stays the primary perceptual number.
_ref = [((50.0000, 2.6772, -79.7751), (50.0000, 0.0000, -82.7485), 2.0425),
        ((50.0000, 3.1571, -77.2803), (50.0000, 0.0000, -82.7485), 2.8615),
        ((50.0000, 2.8361, -74.0200), (50.0000, 0.0000, -82.7485), 3.4412),
        ((50.0000, -1.3802, -84.2814), (50.0000, 0.0000, -82.7485), 1.0000),
        ((50.0000, 2.4900, -0.0010), (50.0000, -2.4900, 0.0009), 7.1792),
        ((60.2574, -34.0099, 36.2677), (60.4626, -34.1751, 39.4387), 1.2644),
        ((2.0776, 0.0795, -1.1350), (0.9033, -0.0636, -0.5514), 0.9082)]
_A = np.array([p[0] for p in _ref]); _B = np.array([p[1] for p in _ref])
_E = np.array([p[2] for p in _ref])
_got = delta_e00_lab(_A, _B)
DE00_OK = bool(np.abs(_got - _E).max() < 1e-2)
print("CIEDE2000 self-test:")
for g, e in zip(_got, _E):
    print(f"   computed {g:8.4f}   reference {e:8.4f}   |diff| {abs(g-e):.2e}")
print("  ->", "PASS" if DE00_OK else
      "CHECK — deltaE00 disagrees with the reference; treat deltaE76 as the primary perceptual metric")

In [ ]:
# ---------------- shape-level evaluation context (XYZ-only, colour-blind, built once) ----------
def local_roughness(rgb, knn_idx):
    '''s(C) = mean_i mean_{j in kNN_xyz(i)} ||c_i - c_j||_2 ; neighbourhoods from XYZ only.'''
    return float(np.linalg.norm(rgb[knn_idx] - rgb[:, None, :], axis=-1).mean())


def edge_strength(rgb, knn_idx):
    '''e_i = max_{j in kNN_xyz(i)} ||c_i - c_j||_2 — local colour contrast, XYZ neighbourhoods.'''
    return np.linalg.norm(rgb[knn_idx] - rgb[:, None, :], axis=-1).max(1)


def hist_chi2(a, b, bins=8):
    ha = np.histogramdd(np.clip(a, 0, 1), bins=bins, range=[(0, 1)] * 3)[0].ravel()
    hb = np.histogramdd(np.clip(b, 0, 1), bins=bins, range=[(0, 1)] * 3)[0].ravel()
    ha = ha / max(ha.sum(), 1); hb = hb / max(hb.sum(), 1)
    return 0.5 * float(np.sum((ha - hb) ** 2 / (ha + hb + 1e-12)))


_SWD_DIRS = None
def swd(A, B, n_proj=64):
    '''Sliced Wasserstein-1 between two equal-size point sets. Directions are FIXED across the whole
    notebook so every model is scored against the same projections.'''
    global _SWD_DIRS
    if _SWD_DIRS is None:
        P = np.random.default_rng(12345).normal(size=(3, n_proj))
        _SWD_DIRS = P / np.linalg.norm(P, axis=0, keepdims=True)
    a = np.sort(A @ _SWD_DIRS, axis=0); b = np.sort(B @ _SWD_DIRS, axis=0)
    return float(np.abs(a - b).mean())


class ShapeCtx:
    '''Everything a metric needs about one validation object. Depends ONLY on ground truth, so it is
    identical for every model — that is what makes the comparison paired.'''

    def __init__(self, xyz, rgb, k=8, n_regions=32, seed=0):
        self.xyz, self.rgb = xyz.astype(np.float64), rgb.astype(np.float64)
        self.lab = srgb_to_lab(self.rgb)
        self.knn = cKDTree(self.xyz).query(self.xyz, k=k + 1)[1][:, 1:]        # XYZ only
        self.rough_gt = local_roughness(self.rgb, self.knn)
        seeds = fps_idx(self.xyz.astype(np.float32), min(n_regions, len(xyz)))  # XYZ only, colour-blind
        self.cell = cdist(self.xyz, self.xyz[seeds]).argmin(1)
        self.cell_w = np.bincount(self.cell, minlength=len(seeds)).astype(float)
        self.n_cells = len(seeds)
        self.lab_cell_gt = srgb_to_lab(self.cell_means(self.rgb))
        # colour-edge reference: threshold comes from GROUND TRUTH and is shared by both models
        self.edge_gt = edge_strength(self.rgb, self.knn)
        self.edge_thr = max(float(np.percentile(self.edge_gt, 90)), 1e-6)
        self.edge_mask_gt = self.edge_gt > self.edge_thr
        # chance level for region_dE: the same colour multiset, spatially scrambled
        perm = np.random.default_rng(seed).permutation(len(xyz))
        self.region_chance = self.region_dE(self.rgb[perm])

    def cell_means(self, rgb):
        s = np.zeros((self.n_cells, 3))
        np.add.at(s, self.cell, rgb)
        return s / np.maximum(self.cell_w, 1)[:, None]

    def region_dE(self, rgb):
        '''Point-count-weighted mean ΔE76 between per-cell mean colours. Cells are the XYZ-only
        FPS-Voronoi partition, so this asks whether each colour landed on the right geometry.'''
        d = np.linalg.norm(srgb_to_lab(self.cell_means(rgb)) - self.lab_cell_gt, axis=1)
        return float(np.average(d, weights=self.cell_w))

    def edge_iou(self, rgb):
        mp = edge_strength(rgb, self.knn) > self.edge_thr
        union = int((mp | self.edge_mask_gt).sum())
        if not self.edge_mask_gt.any() or union == 0:
            return float("nan")
        return float((mp & self.edge_mask_gt).sum() / union)


def colour_metrics(ctx, rgb_pred, per_point=False):
    rgb_pred = np.clip(np.asarray(rgb_pred, float), 0, 1)
    lab_p = srgb_to_lab(rgb_pred)
    d76 = np.linalg.norm(lab_p - ctx.lab, axis=1)
    d00 = delta_e00_lab(ctx.lab, lab_p)
    err = rgb_pred - ctx.rgb
    mse = float((err ** 2).mean())
    rough_p = local_roughness(rgb_pred, ctx.knn)
    reg = ctx.region_dE(rgb_pred)
    # a near-single-coloured object makes the chance level degenerate -> the ratio means nothing
    reg_norm = reg / ctx.region_chance if ctx.region_chance > 1.0 else float("nan")
    out = dict(rgb_mae=float(np.abs(err).mean()), rgb_mse=mse,
               psnr=float(10 * np.log10(1.0 / max(mse, 1e-12))),
               deltaE76=float(d76.mean()), deltaE00=float(d00.mean()),
               deltaE76_p95=float(np.percentile(d76, 95)),
               local_consistency_absdiff=abs(rough_p - ctx.rough_gt),
               # a perfectly flat object has s(C_0) = 0, so the ratio is 0/0 -> undefined, not 0
               local_consistency_ratio=(rough_p / ctx.rough_gt if ctx.rough_gt > 1e-6
                                        else float("nan")),
               hist_chi2=hist_chi2(rgb_pred, ctx.rgb),
               swd_lab=swd(lab_p, ctx.lab),
               region_dE=reg, region_dE_norm=reg_norm, region_dE_chance=ctx.region_chance,
               colour_edge_iou=ctx.edge_iou(rgb_pred))
    if per_point:
        out["_dE76_pp"], out["_dE00_pp"] = d76, d00
    return out


def geometry_metrics(ctx, xyz_pred):
    xyz_pred = np.asarray(xyz_pred, float)
    err = xyz_pred - ctx.xyz
    d = cdist(xyz_pred, ctx.xyz)
    d1, d2 = d.min(1), d.min(0)
    nn = d.argmin(1)
    return dict(x0_mse=float((err ** 2).mean()),
                x0_p2p=float(np.linalg.norm(err, axis=1).mean()),
                x0_p2p_p95=float(np.percentile(np.linalg.norm(err, axis=1), 95)),
                chamfer_l1=float(d1.mean() + d2.mean()),
                chamfer_l2=float((d1 ** 2).mean() + (d2 ** 2).mean()),
                colored_chamfer_dE=float(np.linalg.norm(ctx.lab - ctx.lab[nn], axis=1).mean()))


# lower-is-better unless listed here
HIGHER_IS_BETTER = {"psnr", "colour_edge_iou"}
TARGET_ONE = {"local_consistency_ratio"}          # neither direction: 1.0 is correct
GEO_METRICS = ["eps_mse", "x0_mse", "x0_p2p", "chamfer_l1", "chamfer_l2", "colored_chamfer_dE"]
COL_METRICS = ["eps_mse", "rgb_mae", "rgb_mse", "psnr", "deltaE76", "deltaE00",
               "local_consistency_absdiff", "hist_chi2", "swd_lab",
               "region_dE", "region_dE_norm", "colour_edge_iou"]
PRIMARY = {"geometry": "eps_mse", "colour": "eps_mse"}
PRIMARY_SECONDARY = {"geometry": "x0_mse", "colour": "deltaE00"}


def direction(m):
    return "higher" if m in HIGHER_IS_BETTER else ("target1" if m in TARGET_ONE else "lower")


def arrow(m):
    return {"higher": "↑", "lower": "↓", "target1": "→1"}[direction(m)]


t0 = time.time()
VAL_CTX = [ShapeCtx(DATA["val"]["xyz"][i], DATA["val"]["rgb"][i], k=KNN_K,
                    n_regions=N_REGIONS, seed=1000 + i) for i in range(NUM_VAL_SHAPES)]
print(f"built {len(VAL_CTX)} validation contexts in {time.time()-t0:.1f}s "
      f"({N_REGIONS} XYZ-only regions, k={KNN_K} XYZ neighbours each)")
_ch = np.array([c.region_chance for c in VAL_CTX])
print(f"region_dE chance level (ΔE76 of a colour-scrambled GT): min {_ch.min():.1f}  "
      f"median {np.median(_ch):.1f}  max {_ch.max():.1f}")
if (_ch <= 1.0).any():
    print(f"  NOTE: {(_ch <= 1.0).sum()} object(s) are close to single-coloured; region_dE_norm is "
          f"reported as NaN for them and excluded from the averages.")

# ---- self-consistency: feeding a metric the ground truth must score perfectly ----
# Ratio-shaped metrics are undefined on a perfectly flat object (0/0); the policy throughout is to
# return NaN and let the averages skip that object, never to silently report a 0 or a huge number.
def _isnan(v):
    return v != v


for _c in VAL_CTX:
    _m = colour_metrics(_c, _c.rgb)
    assert _m["rgb_mae"] == 0 and _m["deltaE76"] == 0 and _m["deltaE00"] == 0, _m
    assert _m["region_dE"] < 1e-9 and _m["hist_chi2"] < 1e-12 and _m["swd_lab"] < 1e-9, _m
    assert _m["local_consistency_absdiff"] == 0.0, _m
    assert _m["local_consistency_ratio"] == 1.0 or (_isnan(_m["local_consistency_ratio"])
                                                    and _c.rough_gt <= 1e-6), _m
    assert _m["region_dE_norm"] == 0.0 or _isnan(_m["region_dE_norm"]), _m
    assert _m["colour_edge_iou"] == 1.0 or (_isnan(_m["colour_edge_iou"])
                                            and not _c.edge_mask_gt.any()), _m
    _g = geometry_metrics(_c, _c.xyz)
    assert _g["x0_mse"] == 0 and _g["chamfer_l1"] == 0 and _g["colored_chamfer_dE"] == 0, _g

_n_edge = int(sum(c.edge_mask_gt.any() for c in VAL_CTX))
_n_flat = int(sum(c.rough_gt <= 1e-6 for c in VAL_CTX))
print(f"metric self-consistency on an exact prediction: PASS for all {len(VAL_CTX)} objects "
      f"(every error identically 0, every ratio exactly 1)")
print(f"{_n_edge}/{len(VAL_CTX)} objects carry colour edges" +
      (f"; {_n_flat} are perfectly flat" if _n_flat else "") +
      ". Ratio metrics (region_dE_norm, colour_edge_iou, local_consistency_ratio) are NaN and "
      "excluded\nfor objects where they are undefined, rather than distorting the averages.")

## 12 · Tiny-set overfit experiment  ← **read the verdict before going further**

A diagnostic, **not a result**. Each of the four models is trained on `SANITY_CLOUDS` (≈6) training
objects for `SANITY_STEPS` full-batch steps. A model that cannot drive the loss down on six memorised
clouds has a broken noise pipeline, a broken timestep path or a broken architecture — and nothing
after this section would mean anything.

**Two reference points matter, and they point in opposite directions.**

*The trivial noise predictor* $\hat\epsilon\equiv0$ scores exactly `1.0` at every $t$, because
$\epsilon\sim\mathcal N(0,I)$.

*But $\epsilon$-MSE alone is a misleading progress signal*, because the task is **easiest at high
$t$**: as $\bar\alpha_t\to0$, $\epsilon=(x_t-\sqrt{\bar\alpha_t}x_0)/\sqrt{1-\bar\alpha_t}\to x_t$,
so a model can simply echo its own input and score near zero without knowing anything about the data.
The uniform-$t$ average is dominated by exactly that regime. Conversely at **low $t$** the same
formula divides by $\sqrt{1-\bar\alpha_t}\approx0.09$, so recovering $\epsilon$ demands knowing $x_0$
to within about a ninth of the noise scale — that is where real learning shows.

So the honest check is in $x_0$ space. For each probe timestep we also reconstruct

$$\hat x_0^{\text{model}}\ \text{from}\ \hat\epsilon,
\qquad
\hat x_0^{\text{trivial}}\ \text{from}\ \hat\epsilon\equiv0\ \ (=x_t/\sqrt{\bar\alpha_t}),$$

and report the **gain** $\;1-\mathrm{MSE}(\hat x_0^{\text{model}})/\mathrm{MSE}(\hat x_0^{\text{trivial}})$
— the fraction of the "assume it isn't noisy" error the model actually removes. A model gaming the
high-$t$ shortcut scores a gain near 0.

**Pass criteria** (all four, for all four models):

1. `drop_ok` — loss falls below 70 % of its opening level;
2. `beats_trivial_ok` — final loss below 0.85, i.e. clear of the $\hat\epsilon\equiv0$ predictor;
3. `t_profile_ok` — probe loss at $t{=}750$ < $t{=}250$ < $t{=}50$, the canonical
   $\epsilon$-prediction difficulty profile. A flat profile means the timestep is not reaching the
   network or the schedule is mis-wired;
4. `x0_gain_ok` — $x_0$ gain at $t{=}250$ exceeds 0.15, i.e. the model genuinely denoises rather
   than echoing its input.

If it prints `SANITY CHECK FAILED`, **leave `RUN_FULL_EXPERIMENT = False`** and debug in this order:
§8's assertions → §6's normalisation ranges → raise `SANITY_STEPS` (1500 → 3000) or `WIDTH`.

In [ ]:
SANITY_LR = 1e-3            # SANITY_CLOUDS / SANITY_STEPS live in §1
PROBE_TS  = (50, 250, 750)


def _model_forward(model, key, x_t, t, G0, C0):
    spec = MODEL_SPECS[key]
    cond = None if spec["cond"] is None else (C0 if spec["cond"] == "C0" else G0)
    coords = x_t if uses_local(key) else None      # XYZ (=G_t) only, geometry pair only
    return model(x_t, t, cond, coords)


def tiny_overfit(key, n_clouds=SANITY_CLOUDS, steps=SANITY_STEPS, lr=SANITY_LR, log_every=25):
    set_seed(SEED)
    model = build_model(key).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    G0 = DATA["train"]["G0"][:n_clouds].to(DEV)
    C0 = DATA["train"]["C0"][:n_clouds].to(DEV)
    x0 = G0 if MODEL_SPECS[key]["target"] == "G0" else C0

    pg = torch.Generator().manual_seed(4242)
    probe_noise = torch.randn(n_clouds, NUM_POINTS, 3, generator=pg).to(DEV)
    losses, probe = [], {t: [] for t in PROBE_TS}
    for s in range(steps):
        g = torch.Generator().manual_seed(SEED * 7919 + s)
        t = torch.randint(0, T, (n_clouds,), generator=g).to(DEV)
        noise = torch.randn(n_clouds, NUM_POINTS, 3, generator=g).to(DEV)
        model.train()
        loss = F.mse_loss(_model_forward(model, key, DIF.q_sample(x0, t, noise), t, G0, C0), noise)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if s % log_every == 0 or s == steps - 1:
            model.eval()
            with torch.no_grad():
                for tt in PROBE_TS:
                    tv = torch.full((n_clouds,), tt, dtype=torch.long, device=DEV)
                    x_t = DIF.q_sample(x0, tv, probe_noise)
                    eps = _model_forward(model, key, x_t, tv, G0, C0)
                    e_mse = F.mse_loss(eps, probe_noise).item()
                    x0_m = F.mse_loss(DIF.x0_from_eps(x_t, tv, eps), x0).item()
                    x0_triv = F.mse_loss(DIF.x0_from_eps(x_t, tv, torch.zeros_like(eps)), x0).item()
                    probe[tt].append((s, e_mse, x0_m, x0_triv))
    return dict(losses=np.array(losses),
                probe={k: np.array(v) for k, v in probe.items()})   # cols: step, eps, x0, x0_trivial


def probe_last(h, tt, col):
    '''col: 1 = eps-MSE, 2 = x0-MSE (model), 3 = x0-MSE (trivial eps=0 predictor).'''
    return float(h["probe"][tt][-1, col])


def x0_gain(h, tt):
    '''Fraction of the "assume it is not noisy" reconstruction error the model removes.'''
    triv = probe_last(h, tt, 3)
    return float("nan") if triv <= 0 else 1.0 - probe_last(h, tt, 2) / triv


# The opening window must be SHORT: at the real config the loss falls most of the way inside the
# first ~50 steps, so a wide opening window measures the post-drop level and makes the reduction
# criterion look like a failure.
_OPEN = min(5, SANITY_STEPS)
_EDGE = max(10, SANITY_STEPS // 20)
if RUN_SANITY_CHECK:
    SANITY = {}
    for k in MODEL_SPECS:
        t0 = time.time()
        SANITY[k] = tiny_overfit(k)
        L = SANITY[k]["losses"]
        print(f"  {k:4s} done in {time.time()-t0:5.1f}s   "
              f"loss {L[:_OPEN].mean():.4f} -> {L[-_EDGE:].mean():.4f}", flush=True)
else:
    SANITY = None
    print("RUN_SANITY_CHECK is False — skipped.")

In [ ]:
if SANITY is not None:
    rows = []
    for k, h in SANITY.items():
        init = float(h["losses"][:_OPEN].mean())
        fin = float(h["losses"][-_EDGE:].mean())
        p50, p250, p750 = (probe_last(h, t, 1) for t in (50, 250, 750))
        gain = x0_gain(h, 250)
        c1 = fin < 0.70 * init
        c2 = fin < 0.85
        c3 = p750 < p250 < p50
        c4 = gain > 0.15
        rows.append(dict(model=k, loss_start=init, loss_end=fin, ratio=fin / init,
                         eps_t50=p50, eps_t250=p250, eps_t750=p750,
                         x0_gain_t50=x0_gain(h, 50), x0_gain_t250=gain,
                         drop_ok=c1, beats_trivial_ok=c2, t_profile_ok=c3, x0_gain_ok=c4,
                         passed=bool(c1 and c2 and c3 and c4)))
    SANITY_TABLE = pd.DataFrame(rows)
    SANITY_TABLE.to_csv(os.path.join(TAB_DIR, "sanity_check.csv"), index=False)
    print(SANITY_TABLE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\neps-MSE reference: predicting eps = 0 scores exactly 1.0 at EVERY t.")
    print("eps-MSE is expected to FALL as t rises (at high t, eps ~ x_t, so the net can echo its "
          "input);\nthat is why the x0 gain, not the eps loss, is the criterion that shows real "
          "learning.")

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.1))
    for k, h in SANITY.items():
        w = max(1, len(h["losses"]) // 60)
        sm = np.convolve(h["losses"], np.ones(w) / w, mode="valid")
        axes[0].plot(sm, label=k, lw=1.2)
        axes[1].plot(h["probe"][50][:, 0], h["probe"][50][:, 1], lw=1.2, label=f"{k} t=50")
        axes[2].plot(h["probe"][250][:, 0], 1 - h["probe"][250][:, 2] / h["probe"][250][:, 3],
                     lw=1.2, label=k)
    axes[0].axhline(1.0, color="k", ls="--", lw=.8); axes[0].axhline(0.85, color="g", ls=":", lw=.8)
    axes[0].set_title("tiny-set training loss (uniform t)"); axes[0].set_xlabel("step")
    axes[0].set_ylabel(r"$\epsilon$-MSE"); axes[0].legend(fontsize=7)
    axes[1].axhline(1.0, color="k", ls="--", lw=.8)
    axes[1].set_title(r"$\epsilon$-MSE at t=50 (the hard end)"); axes[1].set_xlabel("step")
    axes[1].legend(fontsize=7)
    axes[2].axhline(0.15, color="g", ls=":", lw=.8); axes[2].axhline(0.0, color="k", lw=.8)
    axes[2].set_title(r"$x_0$ recovery gain at t=250 (pass > 0.15)"); axes[2].set_xlabel("step")
    axes[2].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "12_sanity.png")); plt.show(); plt.close(fig)

    SANITY_PASSED = bool(SANITY_TABLE.passed.all())
    print("\n" + "=" * 74)
    if SANITY_PASSED:
        print("SANITY CHECK PASSED")
        print("  All four models drive the denoising loss down on a tiny memorised set.")
        print("  -> set RUN_FULL_EXPERIMENT = True in §1, re-run §1, then run §13 onwards.")
    else:
        bad = SANITY_TABLE.loc[~SANITY_TABLE.passed, "model"].tolist()
        print("SANITY CHECK FAILED — inspect loss curve before proceeding")
        print(f"  failing models: {bad}")
        print("  Do NOT enable RUN_FULL_EXPERIMENT yet. Debug order: §8 assertions -> §6 "
              "normalisation ranges -> raise SANITY_STEPS or WIDTH.")
    print("=" * 74)

## 13 · Training utilities

The mechanics that make the comparison a *paired* one:

* **Identical batches.** The shape permutation for epoch *e* comes from `default_rng([SEED, e])`, so
  `Model G` and `Model G+C` iterate over the same objects in the same order.
* **Identical noise and timesteps.** Step *(e, s)* seeds a fresh `torch.Generator` from
  `SEED, e, s`, drawing $t$ and $\epsilon$ from it. The two models of a pair therefore train on
  byte-identical $(x_t, t, \epsilon)$ triples. This removes sampling noise from the comparison
  instead of merely averaging it away.
* **Identical initial RNG state.** `build_model` re-seeds before construction.
* **A frozen validation bank.** `VAL_BANK_REPEATS` stratified $(t,\epsilon)$ draws per validation
  object, generated once and reused by every model at every epoch, so the val-loss curves are
  comparable epoch-to-epoch *and* model-to-model.

Checkpoints are written **after every epoch** (`RESUME=True` picks up where an interrupted Kaggle
session stopped; a checkpoint whose stored config no longer matches §1 is rejected rather than
silently reused). `TIME_BUDGET_MIN_PER_MODEL` stops a model early but still saves and evaluates it.

In [ ]:
def epoch_batches(n, bs, epoch):
    perm = np.random.default_rng([SEED, epoch]).permutation(n)
    return [perm[i:i + bs] for i in range(0, n, bs)]


def draw_t_noise(bs, epoch, step):
    g = torch.Generator().manual_seed((SEED * 1000003 + epoch * 10007 + step) % (2 ** 31 - 1))
    t = torch.randint(0, T, (bs,), generator=g)
    noise = torch.randn(bs, NUM_POINTS, 3, generator=g)
    return t.to(DEV), noise.to(DEV)


def build_val_bank(repeats, seed):
    '''Stratified (t, noise) draws — same bank for all four models, frozen for the whole run.'''
    n, M = NUM_VAL_SHAPES, NUM_VAL_SHAPES * repeats
    g = torch.Generator().manual_seed(seed)
    u = (torch.arange(M).float() + torch.rand(M, generator=g)) / M
    t = (u * T).long().clamp(0, T - 1)
    p = torch.randperm(M, generator=g)
    return dict(t=t[p].to(DEV),
                noise=torch.randn(M, NUM_POINTS, 3, generator=g).to(DEV),
                shape=torch.arange(n).repeat(repeats))


VAL_BANK = build_val_bank(VAL_BANK_REPEATS, seed=SEED + 555)
print(f"frozen val bank: {len(VAL_BANK['t'])} (shape, t, noise) triples, "
      f"t spans {int(VAL_BANK['t'].min())}..{int(VAL_BANK['t'].max())}")


@torch.no_grad()
def val_loss(model, key, bank=None, chunk=16):
    bank = VAL_BANK if bank is None else bank
    model.eval(); tot, cnt = 0.0, 0
    for s in range(0, len(bank["t"]), chunk):
        sl = slice(s, s + chunk)
        idx = bank["shape"][sl]
        G0 = DATA["val"]["G0"][idx].to(DEV); C0 = DATA["val"]["C0"][idx].to(DEV)
        x0 = G0 if MODEL_SPECS[key]["target"] == "G0" else C0
        t, noise = bank["t"][sl], bank["noise"][sl]
        eps = _model_forward(model, key, DIF.q_sample(x0, t, noise), t, G0, C0)
        tot += F.mse_loss(eps, noise, reduction="sum").item(); cnt += noise.numel()
    return tot / cnt


def _train_meta():
    return dict(seed=SEED, category=CATEGORY, n_points=NUM_POINTS, n_train=NUM_TRAIN_SHAPES,
                n_val=NUM_VAL_SHAPES, bs=BATCH_SIZE, lr=LEARNING_RATE, wd=WEIGHT_DECAY,
                epochs=NUM_EPOCHS, T=T, schedule=BETA_SCHEDULE, width=WIDTH, blocks=N_BLOCKS,
                local=bool(USE_LOCAL_GEOMETRY_BACKBONE), lr_sched=LR_SCHEDULE,
                source=DATA_ORIGIN, subsample=SUBSAMPLE)


def train_model(key, epochs=None, verbose_every=None):
    epochs = NUM_EPOCHS if epochs is None else epochs
    verbose_every = max(1, epochs // 12) if verbose_every is None else verbose_every
    meta = _train_meta()
    path = os.path.join(CKPT_DIR, f"model_{key.replace('+','_')}.pt")

    model = build_model(key).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=LEARNING_RATE / 100)
             if LR_SCHEDULE == "cosine" else None)
    start, history = 0, []

    if RESUME and not FORCE_RETRAIN and os.path.exists(path):
        try:
            z = torch.load(path, map_location=DEV, weights_only=False)
            if z.get("meta") == meta:
                model.load_state_dict(z["model"]); opt.load_state_dict(z["opt"])
                if sched is not None and z.get("sched") is not None:
                    sched.load_state_dict(z["sched"])
                start, history = z["epoch"] + 1, z["history"]
                print(f"  [{key}] resumed from epoch {start}/{epochs}")
            else:
                print(f"  [{key}] checkpoint config differs from §1 -> retraining from scratch")
        except Exception as e:
            print(f"  [{key}] could not load checkpoint ({type(e).__name__}) -> retraining")

    if start >= epochs:
        print(f"  [{key}] already trained for {epochs} epochs (loaded from checkpoint).")
        model.eval()
        hist = pd.DataFrame(history)
        hist.to_csv(os.path.join(HIST_DIR, f"history_{key.replace('+','_')}.csv"), index=False)
        return model, hist

    n = NUM_TRAIN_SHAPES
    t_start, stopped = time.time(), None
    for ep in range(start, epochs):
        model.train(); tot, cnt = 0.0, 0
        for si, b in enumerate(epoch_batches(n, BATCH_SIZE, ep)):
            G0 = DATA["train"]["G0"][b].to(DEV); C0 = DATA["train"]["C0"][b].to(DEV)
            x0 = G0 if MODEL_SPECS[key]["target"] == "G0" else C0
            t, noise = draw_t_noise(len(b), ep, si)
            loss = F.mse_loss(_model_forward(model, key, DIF.q_sample(x0, t, noise), t, G0, C0), noise)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b); cnt += len(b)
        if sched is not None:
            sched.step()
        vl = val_loss(model, key)
        history.append(dict(epoch=ep, train_loss=tot / cnt, val_loss=vl,
                            lr=opt.param_groups[0]["lr"], seconds=time.time() - t_start))
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(),
                        sched=None if sched is None else sched.state_dict(),
                        epoch=ep, history=history, meta=meta), path)
        if ep == start:
            per = time.time() - t_start
            print(f"  [{key}] {per:.1f}s/epoch -> ETA {(epochs-start)*per/60:.1f} min", flush=True)
        if ep % verbose_every == 0 or ep == epochs - 1:
            print(f"  [{key}] ep {ep:4d}/{epochs}  train {tot/cnt:.4f}  val {vl:.4f}", flush=True)
        if TIME_BUDGET_MIN_PER_MODEL and (time.time() - t_start) / 60 > TIME_BUDGET_MIN_PER_MODEL:
            stopped = ep
            print(f"  [{key}] time budget reached at epoch {ep}; stopping early (checkpoint saved)")
            break

    hist = pd.DataFrame(history)
    hist.to_csv(os.path.join(HIST_DIR, f"history_{key.replace('+','_')}.csv"), index=False)
    if stopped is not None:
        hist.attrs["stopped_early_at"] = stopped
    model.eval()
    return model, hist


MODELS, HISTORY = {}, {}


def _guard(key):
    if not RUN_FULL_EXPERIMENT:
        print(f"RUN_FULL_EXPERIMENT is False -> skipping training of {key}.")
        print("Run §12 first; if it prints SANITY CHECK PASSED, set RUN_FULL_EXPERIMENT = True "
              "in §1, re-run §1, then come back.")
        return False
    if SANITY_PASSED is False:
        print("WARNING: §12 reported SANITY CHECK FAILED and you enabled RUN_FULL_EXPERIMENT "
              "anyway. Proceeding, but treat every number below as suspect.")
    elif SANITY_PASSED is None:
        print("NOTE: §12 was not run in this session, so the pipeline is unverified.")
    return True


print("training utilities ready. RUN_FULL_EXPERIMENT =", RUN_FULL_EXPERIMENT)

## 14 · Train Model G — geometry-only

$\hat\epsilon_g=f_G(G_t,t)$. Baseline of Experiment 1.

In [ ]:
if _guard("G"):
    t0 = time.time()
    MODELS["G"], HISTORY["G"] = train_model("G")
    print(f"Model G trained in {(time.time()-t0)/60:.1f} min")

## 15 · Train Model G+C — geometry conditioned on clean colour

$\hat\epsilon_g=f_{G+C}(G_t,C_0,t)$. Same backbone, same batches, same $(t,\epsilon)$ as §14 —
the only difference is three extra input channels carrying $C_0$.

In [ ]:
if _guard("G+C"):
    t0 = time.time()
    MODELS["G+C"], HISTORY["G+C"] = train_model("G+C")
    print(f"Model G+C trained in {(time.time()-t0)/60:.1f} min")

## 16 · Train Model C — colour-only

$\hat\epsilon_c=f_C(C_t,t)$. Baseline of Experiment 2. This model is **structurally geometry-blind**
(verified in §10): pointwise MLP + permutation-invariant global pooling, no neighbourhood of any kind.

In [ ]:
if _guard("C"):
    t0 = time.time()
    MODELS["C"], HISTORY["C"] = train_model("C")
    print(f"Model C trained in {(time.time()-t0)/60:.1f} min")

## 17 · Train Model C+G — colour conditioned on clean geometry

$\hat\epsilon_c=f_{C+G}(C_t,G_0,t)$. Identical to §16 apart from three extra input channels carrying
$G_0$.

In [ ]:
if _guard("C+G"):
    t0 = time.time()
    MODELS["C+G"], HISTORY["C+G"] = train_model("C+G")
    print(f"Model C+G trained in {(time.time()-t0)/60:.1f} min")

MODELS_READY = all(k in MODELS for k in MODEL_SPECS)
print("\nMODELS_READY =", MODELS_READY)
if MODELS_READY:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for ax, (pair, (b, c)) in zip(axes, PAIRS.items()):
        for k, sty in ((b, "-"), (c, "--")):
            h = HISTORY[k]
            ax.plot(h.epoch, h.train_loss, sty, lw=1.0, alpha=.55, label=f"{k} train")
            ax.plot(h.epoch, h.val_loss, sty, lw=1.6, label=f"{k} val")
        ax.axhline(1.0, color="k", ls=":", lw=.8)
        ax.set_title(f"{pair} pair"); ax.set_xlabel("epoch"); ax.set_ylabel(r"$\epsilon$-MSE")
        ax.legend(fontsize=7)
    fig.suptitle("training curves (dotted line = trivial predictor, eps=0)", fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "17_training_curves.png")); plt.show(); plt.close(fig)
    pd.concat([HISTORY[k].assign(model=k) for k in MODELS]).to_csv(
        os.path.join(HIST_DIR, "history_all.csv"), index=False)

## 18 · Quantitative evaluation

Both models of a pair are evaluated on **the same validation objects, at the same timesteps, from the
same noise tensor**. Concretely: for each $(t, r)$ the noise $\epsilon$ is drawn from a generator
seeded by $(t, r)$ alone, $x_t$ is built once, and *both* models are asked to denoise that identical
tensor. Nothing about the comparison depends on which model happened to draw luckier noise.

`eps_mse` is the training objective; $\hat x_0$ is recovered with the §8 formula (clamped to
$[-1,1]$) and scored with the §11 suites. The **overall** row is the equal-weight mean over the
`EVAL_TIMESTEPS` grid; a separate uniform-$t$ number comes from the frozen validation bank.

**Inspect:** the leakage re-test at the end — it re-runs §10's probe on the *trained* weights, so a
model that somehow learned to exploit a modality it should not see would be caught here too.

In [ ]:
EVAL_SEED = SEED + 90210


def df_to_md(df, fmt="{:.4g}", index=False):
    d = df.reset_index() if index else df
    cols = list(d.columns)
    def f(v):
        if isinstance(v, (float, np.floating)):
            return "" if (v != v) else fmt.format(v)
        return str(v)
    out = ["| " + " | ".join(map(str, cols)) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in d.iterrows():
        out.append("| " + " | ".join(f(r[c]) for c in cols) + " |")
    return "\n".join(out)


def rel_improve(b, c, metric):
    '''100 * (L_base - L_cond) / L_base for lower-is-better; sign-flipped for higher-is-better.'''
    if b is None or c is None or b != b or c != c or abs(b) < 1e-15:
        return float("nan")
    d = direction(metric)
    if d == "target1":
        return float("nan")
    return 100.0 * ((c - b) / abs(b) if d == "higher" else (b - c) / abs(b))


@torch.no_grad()
def make_xt(pair, t_val, r):
    '''The corrupted tensor BOTH models of `pair` will see. Depends only on (pair, t, r).'''
    G0 = DATA["val"]["G0"].to(DEV); C0 = DATA["val"]["C0"].to(DEV)
    x0 = G0 if MODEL_SPECS[PAIRS[pair][0]]["target"] == "G0" else C0
    g = torch.Generator().manual_seed(EVAL_SEED + t_val * 1009 + r * 31 + (0 if pair == "geometry" else 7))
    noise = torch.randn(NUM_VAL_SHAPES, NUM_POINTS, 3, generator=g).to(DEV)
    t = torch.full((NUM_VAL_SHAPES,), t_val, dtype=torch.long, device=DEV)
    return G0, C0, x0, t, noise, DIF.q_sample(x0, t, noise)


@torch.no_grad()
def eval_pair_at_t(pair, t_val, repeats):
    rows = []
    for r in range(repeats):
        G0, C0, x0, t, noise, x_t = make_xt(pair, t_val, r)
        for key in PAIRS[pair]:
            eps = _model_forward(MODELS[key], key, x_t, t, G0, C0)
            per_shape_eps = ((eps - noise) ** 2).mean(dim=(1, 2)).cpu().numpy()
            x0h = DIF.x0_from_eps(x_t, t, eps).cpu().numpy()
            for i in range(NUM_VAL_SHAPES):
                rec = dict(pair=pair, model=key, t=t_val, repeat=r, shape=i,
                           shape_id=DATA["val"]["ids"][i], eps_mse=float(per_shape_eps[i]))
                if pair == "geometry":
                    rec.update(geometry_metrics(VAL_CTX[i], x0h[i]))
                else:
                    rec.update(colour_metrics(VAL_CTX[i], (x0h[i] + 1) / 2))
                rows.append(rec)
    return rows


if MODELS_READY:
    t0 = time.time(); rows = []
    for pair in PAIRS:
        for tv in EVAL_TIMESTEPS:
            rows += eval_pair_at_t(pair, tv, EVAL_REPEATS)
            print(f"  {pair:9s} t={tv:4d} done ({time.time()-t0:5.1f}s)", flush=True)
    EVAL_DF = pd.DataFrame(rows)
    EVAL_DF.to_csv(os.path.join(TAB_DIR, "eval_raw.csv"), index=False)
    print(f"\n{len(EVAL_DF)} evaluation rows "
          f"({NUM_VAL_SHAPES} shapes x {len(EVAL_TIMESTEPS)} t x {EVAL_REPEATS} repeats x 2 models "
          f"x 2 experiments)")
else:
    EVAL_DF = None
    print("Models not trained — §18 onwards is skipped. Run §12, then enable RUN_FULL_EXPERIMENT.")

In [ ]:
if EVAL_DF is not None:
    # ---- uniform-t eps-MSE from the frozen validation bank (independent of the t grid) ----
    UNIFORM = {k: val_loss(MODELS[k], k) for k in MODELS}
    print("uniform-t eps-MSE on the frozen validation bank (trivial predictor = 1.0):")
    for p, (b, c) in PAIRS.items():
        print(f"  {p:9s}  {b:4s} {UNIFORM[b]:.5f}   {c:4s} {UNIFORM[c]:.5f}   "
              f"-> {rel_improve(UNIFORM[b], UNIFORM[c], 'eps_mse'):+.2f}%")

    # ---- per-shape aggregation (mean over t grid and repeats) -> the unit for paired statistics ----
    _mcols = [c for c in EVAL_DF.columns
              if c not in ("pair", "model", "t", "repeat", "shape", "shape_id")]
    PER_SHAPE = EVAL_DF.groupby(["pair", "model", "shape", "shape_id"], as_index=False)[_mcols].mean()
    PER_SHAPE.to_csv(os.path.join(TAB_DIR, "per_shape.csv"), index=False)

    BY_MODEL_T = EVAL_DF.groupby(["pair", "model", "t"], as_index=False)[_mcols].mean()
    BY_MODEL = EVAL_DF.groupby(["pair", "model"], as_index=False)[_mcols].mean()
    BY_MODEL["eps_mse_uniform_t"] = BY_MODEL["model"].map(UNIFORM)
    BY_MODEL_T.to_csv(os.path.join(TAB_DIR, "summary_by_model_t.csv"), index=False)
    BY_MODEL.to_csv(os.path.join(TAB_DIR, "summary_by_model.csv"), index=False)

    print("\n--- geometry pair, averaged over the evaluated t grid (all ↓ except where noted) ---")
    print(BY_MODEL[BY_MODEL.pair == "geometry"][["model"] + GEO_METRICS + ["eps_mse_uniform_t"]]
          .to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    print("\n--- colour pair, averaged over the evaluated t grid ---")
    print(BY_MODEL[BY_MODEL.pair == "colour"][["model"] + COL_METRICS + ["eps_mse_uniform_t"]]
          .to_string(index=False, float_format=lambda v: f"{v:.5f}"))

In [ ]:
if EVAL_DF is not None:
    try:
        from scipy.stats import wilcoxon
    except Exception:
        wilcoxon = None

    def paired_stats(pair, metric, df=None):
        '''Paired over validation OBJECTS: both models saw identical noise on each one.'''
        df = PER_SHAPE if df is None else df
        base, cond = PAIRS[pair]
        b = df[(df.pair == pair) & (df.model == base)].set_index("shape")[metric].sort_index()
        c = df[(df.pair == pair) & (df.model == cond)].set_index("shape")[metric].sort_index()
        b, c = b.values.astype(float), c.values.astype(float)
        keep = np.isfinite(b) & np.isfinite(c)      # e.g. region_dE_norm is NaN on near-flat objects
        b, c = b[keep], c[keep]
        if len(b) == 0:
            return dict(pair=pair, metric=metric, direction=direction(metric), baseline=base,
                        conditioned=cond, baseline_value=np.nan, conditioned_value=np.nan,
                        improvement_pct=np.nan, ci_lo=np.nan, ci_hi=np.nan, win_rate=np.nan,
                        n_shapes=0, wilcoxon_p=np.nan)
        d = (c - b) if direction(metric) == "higher" else (b - c)      # positive = conditioned wins
        rng = np.random.default_rng(0)
        idx = rng.integers(0, len(d), (4000, len(d)))
        with np.errstate(divide="ignore", invalid="ignore"):
            boot = 100 * d[idx].mean(1) / np.abs(b[idx].mean(1))
        lo, hi = np.percentile(boot[np.isfinite(boot)], [2.5, 97.5]) if np.isfinite(boot).any() \
            else (np.nan, np.nan)
        p = np.nan
        if wilcoxon is not None and len(d) >= 6 and np.any(d != 0):
            try:
                p = float(wilcoxon(b, c).pvalue)
            except Exception:
                pass
        return dict(pair=pair, metric=metric, direction=direction(metric),
                    baseline=PAIRS[pair][0], conditioned=PAIRS[pair][1],
                    baseline_value=float(b.mean()), conditioned_value=float(c.mean()),
                    improvement_pct=rel_improve(float(b.mean()), float(c.mean()), metric),
                    ci_lo=float(lo), ci_hi=float(hi),
                    win_rate=float((d > 0).mean()), n_shapes=int(len(d)), wilcoxon_p=p)

    PAIRED = pd.DataFrame(
        [paired_stats("geometry", m) for m in GEO_METRICS] +
        [paired_stats("colour", m) for m in COL_METRICS] +
        [paired_stats("colour", m) for m in ("local_consistency_ratio", "deltaE76_p95")])
    PAIRED.to_csv(os.path.join(TAB_DIR, "paired_stats.csv"), index=False)

    show = PAIRED[["pair", "metric", "direction", "baseline_value", "conditioned_value",
                   "improvement_pct", "ci_lo", "ci_hi", "win_rate", "wilcoxon_p"]]
    print("paired over the", NUM_VAL_SHAPES, "validation objects "
          "(improvement% = 100*(base-cond)/base, sign-corrected per direction)\n")
    print(show.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print("\nwin_rate = fraction of validation OBJECTS on which the conditioned model wins "
          "(0.5 = coin flip).")
    print("ci = 95% bootstrap interval on the relative improvement; an interval spanning 0 means "
          "the effect is not resolved at this sample size.")

In [ ]:
if EVAL_DF is not None:
    # ---- §10's leakage test, re-run verbatim on the TRAINED weights ----
    LEAK_TRAINED = leakage_table(MODELS)
    LEAK_TRAINED.to_csv(os.path.join(TAB_DIR, "leakage_test_trained.csv"), index=False)
    print("leakage re-test on trained weights (same function as §10):\n")
    print(LEAK_TRAINED.to_string(index=False))
    assert (LEAK_TRAINED.verdict == "PASS").all(), "leakage detected after training"
    print("\nA trained model cannot have learned to exploit a modality it never receives; this "
          "confirms\nthe wiring survived training and checkpoint round-trips.")

## 19 · Evaluation across diffusion timesteps

One average hides the interesting part: conditioning may only start paying off once the corruption is
severe enough that the noisy modality alone stops being informative. Relative improvement is

$$100\cdot\frac{L_{\text{baseline}}-L_{\text{conditioned}}}{L_{\text{baseline}}}$$

for every ↓ metric, with the sign flipped for ↑ metrics (only `psnr`).

**Inspect:** the *shape* of the improvement-vs-$t$ curve, not just its average. A curve that rises with
$t$ says conditioning substitutes for destroyed information — which is exactly the regime a completion
model operates in.

In [ ]:
def timestep_table(pair, metric):
    '''t | baseline | conditioned | relative improvement %  (column names are the model names, so
    tables for different metrics of the same pair concatenate cleanly).'''
    base, cond = PAIRS[pair]
    d = BY_MODEL_T[BY_MODEL_T.pair == pair]
    b = d[d.model == base].set_index("t")[metric].sort_index()
    c = d[d.model == cond].set_index("t")[metric].sort_index()
    out = pd.DataFrame({"t": b.index.values, base: b.values, cond: c.values})
    out["improvement_%"] = [rel_improve(x, y, metric) for x, y in zip(b.values, c.values)]
    return out


if EVAL_DF is not None:
    TS_TABLES = {}
    for pair, metrics in (("geometry", ["eps_mse", "x0_mse", "chamfer_l1", "colored_chamfer_dE"]),
                          ("colour", ["eps_mse", "deltaE00", "rgb_mae", "psnr", "region_dE_norm",
                                      "colour_edge_iou"])):
        for m in metrics:
            TS_TABLES[(pair, m)] = timestep_table(pair, m)
        block = pd.concat([TS_TABLES[(pair, m)].assign(metric=m) for m in metrics])
        block.to_csv(os.path.join(TAB_DIR, f"timestep_{pair}.csv"), index=False)

    for pair, m in (("geometry", "eps_mse"), ("geometry", "x0_mse"),
                    ("colour", "eps_mse"), ("colour", "deltaE00")):
        print(f"\n--- {pair}: {m} ({arrow(m)}) by timestep ---")
        print(TS_TABLES[(pair, m)].to_string(index=False, float_format=lambda v: f"{v:.5f}"))

In [ ]:
if EVAL_DF is not None:
    specs = [("geometry", ["eps_mse", "x0_mse", "chamfer_l1", "colored_chamfer_dE"]),
             ("colour", ["eps_mse", "deltaE00", "rgb_mae", "psnr", "region_dE_norm",
                         "colour_edge_iou"])]
    for pair, metrics in specs:
        base, cond = PAIRS[pair]
        n = len(metrics)
        fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 5.0), squeeze=False)
        for j, m in enumerate(metrics):
            tt = TS_TABLES[(pair, m)]
            ax = axes[0][j]
            ax.plot(tt.t, tt[base], "o-", lw=1.4, label=base)
            ax.plot(tt.t, tt[cond], "s--", lw=1.4, label=cond)
            ax.set_title(f"{m} {arrow(m)}", fontsize=8); ax.set_xlabel("t")
            if j == 0: ax.legend(fontsize=7)
            ax2 = axes[1][j]
            ax2.bar(tt.t, tt["improvement_%"], width=40,
                    color=["#2f9d66" if v > 0 else "#e0574c" for v in tt["improvement_%"]])
            ax2.axhline(0, color="k", lw=.8); ax2.set_xlabel("t")
            ax2.set_title("improvement %", fontsize=8)
        fig.suptitle(f"{pair} pair — {base} vs {cond} across diffusion timesteps", fontsize=10)
        fig.tight_layout()
        fig.savefig(os.path.join(FIG_DIR, f"19_timestep_{pair}.png"), bbox_inches="tight"); plt.show(); plt.close(fig)

## 20 · Colour metric analysis

The full colour suite for both colour models, plus the diagnostics that let you tell *which* kind of
colour failure you are looking at. Read the metrics **together**:

* `rgb_mae` good but `deltaE00` bad → errors sit where the eye is sensitive (dark/desaturated regions).
* `hist_chi2` / `swd_lab` good but `region_dE` / `region_dE_norm` bad → **right palette, wrong
  places** — the
  model learned the object's colour distribution without learning where each colour belongs.
* `local_consistency_ratio` ≫ 1 → noisy, speckled colour. ≪ 1 → over-smoothed, washed-out colour.
  Both are compatible with a mediocre-but-not-terrible MAE.
* `colour_edge_iou` low while `region_dE` is fine → region means are right but material boundaries
  are misplaced or smoothed away.
* `region_dE_chance` near 0 → the object is nearly single-coloured and `region_dE_norm` is `NaN`
  for it (excluded from the averages rather than silently distorting them).

In [ ]:
if EVAL_DF is not None:
    cols = ["rgb_mae", "rgb_mse", "psnr", "deltaE76", "deltaE00", "deltaE76_p95",
            "local_consistency_absdiff", "local_consistency_ratio", "hist_chi2", "swd_lab",
            "region_dE", "region_dE_norm", "region_dE_chance", "colour_edge_iou"]
    # region_dE_chance is a property of the OBJECT, not a quality score -> no direction arrow
    _ren = {c: (f"{c} {arrow(c)}" if c != "region_dE_chance" else f"{c} (reference)") for c in cols}
    COLOUR_TABLE = BY_MODEL[BY_MODEL.pair == "colour"].set_index("model")[cols].rename(columns=_ren)
    COLOUR_TABLE.to_csv(os.path.join(TAB_DIR, "colour_table.csv"))
    print("colour suite, averaged over the evaluated t grid and", EVAL_REPEATS, "noise draws\n")
    print(COLOUR_TABLE.T.to_string(float_format=lambda v: f"{v:.5f}"))

    gt_rough = float(np.mean([c.rough_gt for c in VAL_CTX]))
    print(f"\nground-truth local colour roughness s(C_0) = {gt_rough:.4f} "
          f"(local_consistency_ratio is s(pred)/s(GT); 1.0 is correct)")
    print("region_dE_norm reference: 0 = colours in exactly the right regions, 1 = no better than a "
          "spatially scrambled copy of the GT palette.")
    print("colour_edge_iou reference: 1 = colour boundaries exactly where GT has them, 0 = none of "
          "them recovered.")

    # per-metric agreement check across the two colour models
    dis = []
    for m in ["rgb_mae", "deltaE00", "hist_chi2", "swd_lab", "region_dE", "region_dE_norm",
              "colour_edge_iou", "local_consistency_absdiff", "psnr"]:
        r = PAIRED[(PAIRED.pair == "colour") & (PAIRED.metric == m)].iloc[0]
        fav = ("undetermined" if r.improvement_pct != r.improvement_pct
               else PAIRS["colour"][1] if r.improvement_pct > 0 else PAIRS["colour"][0])
        dis.append(dict(metric=m, favours=fav,
                        improvement_pct=r.improvement_pct, win_rate=r.win_rate))
    METRIC_AGREEMENT = pd.DataFrame(dis)
    METRIC_AGREEMENT.to_csv(os.path.join(TAB_DIR, "metric_agreement.csv"), index=False)
    print("\nwhich model each colour metric favours (disagreement here is itself a finding):\n")
    print(METRIC_AGREEMENT.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
if EVAL_DF is not None:
    # per-object distribution of the primary colour metrics, paired
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    for ax, m in zip(axes, ["eps_mse", "deltaE00", "region_dE"]):
        d = PER_SHAPE[PER_SHAPE.pair == "colour"]
        b = d[d.model == PAIRS["colour"][0]].sort_values("shape")[m].values
        c = d[d.model == PAIRS["colour"][1]].sort_values("shape")[m].values
        for i in range(len(b)):
            ax.plot([0, 1], [b[i], c[i]], "-", color="#2f9d66" if c[i] < b[i] else "#e0574c",
                    lw=.9, alpha=.75, marker="o", ms=3)
        ax.set_xticks([0, 1]); ax.set_xticklabels(list(PAIRS["colour"]))
        ax.set_title(f"{m} {arrow(m)} per validation object", fontsize=8)
    fig.suptitle("green = conditioning helped this object, red = it hurt", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "20_per_object_colour.png")); plt.show(); plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    for ax, m in zip(axes, ["eps_mse", "x0_mse", "colored_chamfer_dE"]):
        d = PER_SHAPE[PER_SHAPE.pair == "geometry"]
        b = d[d.model == PAIRS["geometry"][0]].sort_values("shape")[m].values
        c = d[d.model == PAIRS["geometry"][1]].sort_values("shape")[m].values
        for i in range(len(b)):
            ax.plot([0, 1], [b[i], c[i]], "-", color="#2f9d66" if c[i] < b[i] else "#e0574c",
                    lw=.9, alpha=.75, marker="o", ms=3)
        ax.set_xticks([0, 1]); ax.set_xticklabels(list(PAIRS["geometry"]))
        ax.set_title(f"{m} {arrow(m)} per validation object", fontsize=8)
    fig.suptitle("geometry pair, per validation object", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "20_per_object_geometry.png")); plt.show(); plt.close(fig)

## 21 · Visual comparisons

Same validation objects, same noise draw as evaluation repeat 0, same camera for every panel.

* **Geometry experiment** — GT / noisy $G_t$ / `Model G` $\hat G_0$ / `Model G+C` $\hat G_0$, all
  painted with the *clean* RGB so that a point landing in the wrong place is visible as a colour in
  the wrong place.
* **Colour experiment** — geometry held fixed at $G_0$; GT colour / noisy $C_t$ / `Model C` $\hat C_0$
  / `Model C+G` $\hat C_0$, rendering the actual reconstructed RGB.

Aggregate metrics hide failure modes; this is where you look for them.

In [ ]:
@torch.no_grad()
def reconstruct(pair, t_val, r=0):
    '''x0-hat for both models of a pair on ALL val shapes, from evaluation repeat r's exact noise.'''
    G0, C0, x0, t, noise, x_t = make_xt(pair, t_val, r)
    out = {"x_t": x_t.cpu().numpy()}
    for key in PAIRS[pair]:
        eps = _model_forward(MODELS[key], key, x_t, t, G0, C0)
        out[key] = DIF.x0_from_eps(x_t, t, eps).cpu().numpy()
    return out


def rank_shapes(pair, metric, t_val=None):
    d = EVAL_DF[(EVAL_DF.pair == pair) & (EVAL_DF.model == PAIRS[pair][1])]
    if t_val is not None:
        d = d[d.t == t_val]
    s = d.groupby("shape")[metric].mean().sort_values(
        ascending=direction(metric) != "higher")
    return s.index.tolist(), s


if EVAL_DF is not None:
    if VIS_T not in EVAL_TIMESTEPS:
        print(f"NOTE: VIS_T={VIS_T} is not in EVAL_TIMESTEPS={EVAL_TIMESTEPS}, so these figures show "
              f"a noise level\n      that has no row in the quantitative tables. Set VIS_T to one of "
              f"them to keep them in step.")
    VIS_IDX = list(range(min(N_VIS_SHAPES, NUM_VAL_SHAPES)))
    REC_G = reconstruct("geometry", VIS_T, 0)
    REC_C = reconstruct("colour", VIS_T, 0)

    panels = []
    for i in VIS_IDX:
        rgb = DATA["val"]["rgb"][i]
        panels += [(f"[{i}] GT geometry", DATA["val"]["xyz"][i], rgb),
                   (f"[{i}] noisy $G_t$, t={VIS_T}", REC_G["x_t"][i], rgb),
                   (f"[{i}] Model G", REC_G["G"][i], rgb),
                   (f"[{i}] Model G+C", REC_G["G+C"][i], rgb)]
    grid3d(panels, ncols=4, figsize_per=2.5, lim=1.35,
           suptitle=f"Experiment 1 — geometry denoising at t={VIS_T} (clean RGB used as paint; "
                    f"axes clipped to ±1.35)",
           path=os.path.join(FIG_DIR, f"21_geometry_t{VIS_T}.png"))

    panels = []
    for i in VIS_IDX:
        xyz = DATA["val"]["xyz"][i]
        panels += [(f"[{i}] GT colour", xyz, DATA["val"]["rgb"][i]),
                   (f"[{i}] noisy $C_t$, t={VIS_T}", xyz, np.clip((REC_C["x_t"][i] + 1) / 2, 0, 1)),
                   (f"[{i}] Model C", xyz, np.clip((REC_C["C"][i] + 1) / 2, 0, 1)),
                   (f"[{i}] Model C+G", xyz, np.clip((REC_C["C+G"][i] + 1) / 2, 0, 1))]
    grid3d(panels, ncols=4, figsize_per=2.5,
           suptitle=f"Experiment 2 — colour denoising at t={VIS_T} (geometry fixed at $G_0$; noisy "
                    f"colours clipped to [0,1] for display only)",
           path=os.path.join(FIG_DIR, f"21_colour_t{VIS_T}.png"))

In [ ]:
if EVAL_DF is not None:
    # one object, both experiments, across the whole t grid — same object, same camera
    i = VIS_IDX[0]
    rows = []
    for tv in EVAL_TIMESTEPS:
        rg, rc = reconstruct("geometry", tv, 0), reconstruct("colour", tv, 0)
        rows.append((tv, rg, rc))
    panels = []
    for tv, rg, _ in rows:
        panels += [(f"t={tv} G", rg["G"][i], DATA["val"]["rgb"][i]),
                   (f"t={tv} G+C", rg["G+C"][i], DATA["val"]["rgb"][i])]
    grid3d(panels, ncols=len(EVAL_TIMESTEPS), figsize_per=1.9, lim=1.35,
           suptitle=f"object [{i}] — geometry reconstruction, Model G (top row pairs) vs Model G+C",
           path=os.path.join(FIG_DIR, "21_geometry_vs_t.png"))
    panels = []
    for tv, _, rc in rows:
        panels += [(f"t={tv} C", DATA["val"]["xyz"][i], np.clip((rc["C"][i] + 1) / 2, 0, 1)),
                   (f"t={tv} C+G", DATA["val"]["xyz"][i], np.clip((rc["C+G"][i] + 1) / 2, 0, 1))]
    grid3d(panels, ncols=len(EVAL_TIMESTEPS), figsize_per=1.9,
           suptitle=f"object [{i}] — colour reconstruction, Model C vs Model C+G",
           path=os.path.join(FIG_DIR, "21_colour_vs_t.png"))

In [ ]:
if EVAL_DF is not None:
    # best / median / worst validation objects, ranked by the conditioned colour model
    order, vals = rank_shapes("colour", "deltaE00", VIS_T)
    picks = [("best", order[0]), ("median", order[len(order) // 2]), ("worst", order[-1])]
    panels = []
    for tag, i in picks:
        panels += [(f"{tag} [{i}] GT", DATA["val"]["xyz"][i], DATA["val"]["rgb"][i]),
                   (f"{tag} Model C", DATA["val"]["xyz"][i], np.clip((REC_C["C"][i] + 1) / 2, 0, 1)),
                   (f"{tag} Model C+G  ΔE00={vals.loc[i]:.1f}",
                    DATA["val"]["xyz"][i], np.clip((REC_C["C+G"][i] + 1) / 2, 0, 1))]
    grid3d(panels, ncols=3, figsize_per=2.5,
           suptitle=f"colour: best / median / worst validation objects at t={VIS_T} "
                    f"(ranked by Model C+G ΔE00)",
           path=os.path.join(FIG_DIR, "21_colour_best_median_worst.png"))
    BMW_COLOUR = pd.DataFrame([dict(rank=t, shape=i, shape_id=DATA["val"]["ids"][i],
                                    deltaE00_cond=float(vals.loc[i])) for t, i in picks])
    BMW_COLOUR.to_csv(os.path.join(TAB_DIR, "best_median_worst_colour.csv"), index=False)
    print(BMW_COLOUR.to_string(index=False))

    order_g, vals_g = rank_shapes("geometry", "x0_mse", VIS_T)
    picks_g = [("best", order_g[0]), ("median", order_g[len(order_g) // 2]), ("worst", order_g[-1])]
    panels = []
    for tag, i in picks_g:
        panels += [(f"{tag} [{i}] GT", DATA["val"]["xyz"][i], DATA["val"]["rgb"][i]),
                   (f"{tag} Model G", REC_G["G"][i], DATA["val"]["rgb"][i]),
                   (f"{tag} Model G+C", REC_G["G+C"][i], DATA["val"]["rgb"][i])]
    grid3d(panels, ncols=3, figsize_per=2.5, lim=1.35,
           suptitle=f"geometry: best / median / worst validation objects at t={VIS_T} "
                    f"(ranked by Model G+C x0-MSE)",
           path=os.path.join(FIG_DIR, "21_geometry_best_median_worst.png"))
    BMW_GEOM = pd.DataFrame([dict(rank=t, shape=i, shape_id=DATA["val"]["ids"][i],
                                  x0_mse_cond=float(vals_g.loc[i])) for t, i in picks_g])
    BMW_GEOM.to_csv(os.path.join(TAB_DIR, "best_median_worst_geometry.csv"), index=False)
    print(BMW_GEOM.to_string(index=False))

In [ ]:
if EVAL_DF is not None and SAVE_PLOTLY:
    # interactive panels — adapted from scripts/toy_part_color.py :: write_panels
    def write_panels(path, panels):
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        fig = make_subplots(rows=1, cols=len(panels), specs=[[{"type": "scene"}] * len(panels)],
                            subplot_titles=[t for t, _, _ in panels])
        for col, (_, xyz, rgb) in enumerate(panels, 1):
            c = ["rgb(%d,%d,%d)" % tuple((np.clip(v, 0, 1) * 255).astype(int)) for v in rgb]
            fig.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="markers",
                                       marker=dict(size=1.8, color=c)), 1, col)
        fig.update_layout(height=520, showlegend=False, margin=dict(l=0, r=0, t=30, b=0))
        for i in range(1, len(panels) + 1):
            fig.layout["scene" if i == 1 else f"scene{i}"].aspectmode = "data"
        fig.write_html(path)

    try:
        i = VIS_IDX[0]
        write_panels(os.path.join(FIG_DIR, f"21_interactive_geometry_t{VIS_T}.html"),
                     [("GT", DATA["val"]["xyz"][i], DATA["val"]["rgb"][i]),
                      (f"noisy t={VIS_T}", REC_G["x_t"][i], DATA["val"]["rgb"][i]),
                      ("Model G", REC_G["G"][i], DATA["val"]["rgb"][i]),
                      ("Model G+C", REC_G["G+C"][i], DATA["val"]["rgb"][i])])
        write_panels(os.path.join(FIG_DIR, f"21_interactive_colour_t{VIS_T}.html"),
                     [("GT", DATA["val"]["xyz"][i], DATA["val"]["rgb"][i]),
                      (f"noisy t={VIS_T}", DATA["val"]["xyz"][i],
                       np.clip((REC_C["x_t"][i] + 1) / 2, 0, 1)),
                      ("Model C", DATA["val"]["xyz"][i], np.clip((REC_C["C"][i] + 1) / 2, 0, 1)),
                      ("Model C+G", DATA["val"]["xyz"][i], np.clip((REC_C["C+G"][i] + 1) / 2, 0, 1))])
        print("interactive panels written to", FIG_DIR)
    except Exception as e:
        print("plotly export skipped:", type(e).__name__, e)

## 22 · Error maps and distributions

Per-point $\Delta E_{00}$ painted onto the cloud, on a **shared colour scale** so the two models are
directly comparable, plus the RGB/Lab distributions that the histogram and sliced-Wasserstein metrics
summarise numerically.

**Inspect:** whether error is spread thinly everywhere (a global palette problem) or concentrated on
particular structures (a placement problem). The former shows up in `hist_chi2`, the latter in
`region_dE` / `region_dE_norm`, and misplaced boundaries in `colour_edge_iou`.

In [ ]:
if EVAL_DF is not None:
    for i in VIS_IDX:
        gt = DATA["val"]["rgb"][i]; xyz = DATA["val"]["xyz"][i]
        pc = np.clip((REC_C["C"][i] + 1) / 2, 0, 1)
        pg = np.clip((REC_C["C+G"][i] + 1) / 2, 0, 1)
        e1 = delta_e00(gt, pc); e2 = delta_e00(gt, pg)
        vmax = float(np.percentile(np.concatenate([e1, e2]), 97))
        fig = plt.figure(figsize=(10.5, 2.8))
        for j, (ttl, rgbv, ev) in enumerate([("GT colour", gt, None),
                                             (f"Model C  mean ΔE00={e1.mean():.1f}", pc, e1),
                                             (f"Model C+G  mean ΔE00={e2.mean():.1f}", pg, e2)]):
            ax = fig.add_subplot(1, 4, j + 1, projection="3d")
            plot_cloud(ax, xyz, rgbv, ttl)
        ax = fig.add_subplot(1, 4, 4, projection="3d")
        sc = plot_cloud(ax, xyz, None, "ΔE00 difference map\n(blue: C+G better)",
                        cmap_vals=e1 - e2, cmap="coolwarm", vmin=-vmax, vmax=vmax)
        fig.colorbar(sc, ax=ax, fraction=.03)
        fig.suptitle(f"object [{i}] {DATA['val']['ids'][i][:18]} — colour error at t={VIS_T}",
                     fontsize=9)
        fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, f"22_error_map_colour_{i}.png"),
                                        bbox_inches="tight"); plt.show(); plt.close(fig)

        fig = plt.figure(figsize=(10.5, 2.8))
        for j, (ttl, ev) in enumerate([("Model C ΔE00", e1), ("Model C+G ΔE00", e2)]):
            ax = fig.add_subplot(1, 2, j + 1, projection="3d")
            sc = plot_cloud(ax, xyz, None, ttl, cmap_vals=ev, cmap="magma", vmin=0, vmax=vmax)
            fig.colorbar(sc, ax=ax, fraction=.03)
        fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, f"22_error_map_absolute_{i}.png"),
                                        bbox_inches="tight"); plt.show(); plt.close(fig)

In [ ]:
if EVAL_DF is not None:
    i = VIS_IDX[0]
    gt = DATA["val"]["rgb"][i]
    pc = np.clip((REC_C["C"][i] + 1) / 2, 0, 1); pg = np.clip((REC_C["C+G"][i] + 1) / 2, 0, 1)
    fig, axes = plt.subplots(1, 4, figsize=(12, 2.8))
    for ch, nm in enumerate("rgb"):
        for arr, lab, sty in ((gt, "GT", "-"), (pc, "Model C", "--"), (pg, "Model C+G", ":")):
            axes[ch].hist(arr[:, ch], bins=48, range=(0, 1), histtype="step", lw=1.4,
                          linestyle=sty, label=lab)
        axes[ch].set_title(f"{nm} channel", fontsize=8); axes[ch].set_xlim(0, 1)
    axes[0].legend(fontsize=7)
    for arr, lab, mk in ((gt, "GT", "o"), (pc, "Model C", "s"), (pg, "Model C+G", "^")):
        L = srgb_to_lab(arr)
        axes[3].scatter(L[:, 1], L[:, 2], s=2, alpha=.35, marker=mk, label=lab)
    axes[3].set_xlabel("a*"); axes[3].set_ylabel("b*"); axes[3].set_title("Lab chroma", fontsize=8)
    axes[3].legend(fontsize=7, markerscale=3)
    fig.suptitle(f"object [{i}] — colour distributions at t={VIS_T} "
                 "(what hist_chi2 / swd_lab measure)", fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "22_colour_distributions.png")); plt.show(); plt.close(fig)

    # error vs t, aggregated
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.0))
    for ax, (pair, m) in zip(axes, [("geometry", "x0_p2p"), ("colour", "deltaE00")]):
        for key in PAIRS[pair]:
            d = BY_MODEL_T[(BY_MODEL_T.pair == pair) & (BY_MODEL_T.model == key)]
            sd = EVAL_DF[(EVAL_DF.pair == pair) & (EVAL_DF.model == key)].groupby("t")[m].std()
            ax.errorbar(d.t, d[m], yerr=sd.values, fmt="o-", capsize=3, lw=1.4, label=key)
        ax.set_xlabel("t"); ax.set_title(f"{pair}: {m} {arrow(m)} vs t (±1 sd over objects)",
                                         fontsize=8)
        ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "22_error_vs_t.png")); plt.show(); plt.close(fig)

## 23 · Aggregate tables and plots

The two headline tables. `Improvement` is the sign-corrected relative change defined in §19; the
bootstrap interval and win rate come from §18's paired statistics.

In [ ]:
if EVAL_DF is not None:
    def headline_row(pair, metric, label):
        r = PAIRED[(PAIRED.pair == pair) & (PAIRED.metric == metric)].iloc[0]
        return dict(Experiment=label, Baseline=r.baseline, Conditioned=r.conditioned,
                    Metric=f"{metric} {arrow(metric)}",
                    **{"Main Baseline Metric": r.baseline_value,
                       "Main Conditioned Metric": r.conditioned_value,
                       "Improvement %": r.improvement_pct,
                       "95% CI": f"[{r.ci_lo:.1f}, {r.ci_hi:.1f}]",
                       "Win rate": r.win_rate, "Wilcoxon p": r.wilcoxon_p})

    HEADLINE = pd.DataFrame([
        headline_row("geometry", "eps_mse", "Geometry denoising (XYZ only -> XYZ+RGB)"),
        headline_row("geometry", "x0_mse", "Geometry denoising — clean-XYZ recovery"),
        headline_row("colour", "eps_mse", "Colour denoising (RGB only -> RGB+XYZ)"),
        headline_row("colour", "deltaE00", "Colour denoising — perceptual ΔE00"),
    ])
    HEADLINE.to_csv(os.path.join(TAB_DIR, "headline.csv"), index=False)
    print(HEADLINE.to_string(index=False, float_format=lambda v: f"{v:.5f}"))

    GEOM_TABLE = (BY_MODEL[BY_MODEL.pair == "geometry"].set_index("model")[GEO_METRICS]
                  .rename(columns={c: f"{c} {arrow(c)}" for c in GEO_METRICS}))
    GEOM_TABLE.to_csv(os.path.join(TAB_DIR, "geometry_table.csv"))
    print("\n--- geometry suite ---")
    print(GEOM_TABLE.to_string(float_format=lambda v: f"{v:.5f}"))
    print("\n--- colour suite (see §20 for the full table) ---")
    print(COLOUR_TABLE[[f"rgb_mae {arrow('rgb_mae')}", f"rgb_mse {arrow('rgb_mse')}",
                        f"deltaE00 {arrow('deltaE00')}", f"psnr {arrow('psnr')}",
                        f"local_consistency_ratio {arrow('local_consistency_ratio')}",
                        f"swd_lab {arrow('swd_lab')}", f"region_dE {arrow('region_dE')}",
                        f"region_dE_norm {arrow('region_dE_norm')}",
                        f"colour_edge_iou {arrow('colour_edge_iou')}"]]
          .to_string(float_format=lambda v: f"{v:.5f}"))

In [ ]:
if EVAL_DF is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    for ax, pair in zip(axes, ["geometry", "colour"]):
        ms = GEO_METRICS if pair == "geometry" else \
            ["eps_mse", "rgb_mae", "deltaE00", "psnr", "hist_chi2", "swd_lab", "region_dE",
             "region_dE_norm", "colour_edge_iou"]
        v = [PAIRED[(PAIRED.pair == pair) & (PAIRED.metric == m)].iloc[0].improvement_pct for m in ms]
        ax.barh(range(len(ms)), v, color=["#2f9d66" if x > 0 else "#e0574c" for x in v])
        ax.set_yticks(range(len(ms))); ax.set_yticklabels([f"{m} {arrow(m)}" for m in ms], fontsize=7)
        ax.axvline(0, color="k", lw=.9)
        ax.set_xlabel("relative improvement of the conditioned model (%)")
        ax.set_title(f"{pair}: {PAIRS[pair][0]} → {PAIRS[pair][1]}", fontsize=9)
    fig.suptitle("positive = conditioning helped (sign already corrected per metric direction)",
                 fontsize=9)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "23_improvement_summary.png")); plt.show(); plt.close(fig)

## 24 · Automatic experimental report

Written from the CSVs this run produced — nothing is hard-coded, and every claim is generated by a
branch on an actual number. Run it after §14–23. Output: `results/experiment_report.md` (required) and
`results/experiment_report.html`.

The wording rules baked in follow the interpretation policy for this study: a win means *clean colour
carries useful predictive information for recovering corrupted geometry* (or vice versa) **under this
controlled denoising setup** — never that a joint completion framework beats a geometry-only one. A
null result is reported as a null result with its plausible causes, not as a refutation.

In [ ]:
def _load(name):
    p = os.path.join(TAB_DIR, name)
    return pd.read_csv(p) if os.path.exists(p) else None


def generate_report():
    par = _load("paired_stats.csv")
    if par is None:
        return ("# Experiment report\n\nNo results found in `%s`.\n\nRun §12, set "
                "`RUN_FULL_EXPERIMENT = True` in §1, then run §14 through §23 before generating "
                "the report.\n" % TAB_DIR)

    hl, bym, bymt = _load("headline.csv"), _load("summary_by_model.csv"), _load("summary_by_model_t.csv")
    colt, geot = _load("colour_table.csv"), _load("geometry_table.csv")
    agree, pars = _load("metric_agreement.csv"), _load("model_parameters.csv")
    norm, snrt = _load("normalization_stats.csv"), _load("snr_by_timestep.csv")
    leak, san = _load("leakage_test_trained.csv"), _load("sanity_check.csv")
    bmc, bmg = _load("best_median_worst_colour.csv"), _load("best_median_worst_geometry.csv")
    tsg, tsc = _load("timestep_geometry.csv"), _load("timestep_colour.csv")

    def row(pair, metric):
        m = par[(par.pair == pair) & (par.metric == metric)]
        return None if len(m) == 0 else m.iloc[0]

    g_eps, c_eps = row("geometry", "eps_mse"), row("colour", "eps_mse")
    g_x0, c_de = row("geometry", "x0_mse"), row("colour", "deltaE00")
    g_win = bool(g_eps is not None and g_eps.improvement_pct > 0)
    c_win = bool(c_eps is not None and c_eps.improvement_pct > 0)

    def resolved(r):
        '''Is the effect resolved at this sample size? (bootstrap CI excludes 0)'''
        return bool(r is not None and r.ci_lo == r.ci_lo and (r.ci_lo > 0 or r.ci_hi < 0))

    def verdict(r, name_b, name_c):
        if r is None:
            return "not measured"
        s = (f"{name_c} {'improves on' if r.improvement_pct > 0 else 'is worse than'} {name_b} by "
             f"**{abs(r.improvement_pct):.2f}%** ({r.baseline_value:.5f} → {r.conditioned_value:.5f}), "
             f"winning on {r.win_rate*100:.0f}% of the {int(r.n_shapes)} validation objects; "
             f"95% bootstrap CI [{r.ci_lo:.1f}%, {r.ci_hi:.1f}%]")
        if r.wilcoxon_p == r.wilcoxon_p:
            s += f", Wilcoxon p={r.wilcoxon_p:.2g}"
        return s + ("." if resolved(r) else
                    ". **The interval spans 0, so this effect is not resolved at this sample size.**")

    L = []
    A = L.append
    A(f"# Geometry ↔ Colour Mutual Denoising — feasibility report\n")
    A(f"*Generated automatically from the run in `{RESULTS_DIR}`. Every number below was produced by "
      f"this execution; nothing is pre-filled.*\n")
    if IS_SYNTHETIC:
        A("> **WARNING — SYNTHETIC FALLBACK DATA.** The real dataset was unreachable, so this run used "
          "procedurally generated boxes. Treat every result as a pipeline smoke test only.\n")

    # 1
    A("## 1. Objective and hypothesis\n")
    A("Long-term goal: a joint 6-D diffusion model for colored point-cloud completion, "
      "$p_i=[x_i,y_i,z_i,r_i,g_i,b_i]$. The hypothesis tested here is narrow and prior to that: "
      "**geometry and colour carry mutually useful information during denoising.** Two symmetric "
      "controlled experiments ask whether conditioning on the *clean* other modality lowers the "
      "noise-prediction error. This is not a completion system and establishes nothing about one.\n")

    # 2
    A("## 2. Dataset, category, sample sizes\n")
    A(f"- source: `{DATA_ORIGIN}`" + (f" (`{HF_DATASET}`, `labeled_s3/{SYNSET}`)"
                                      if DATA_ORIGIN == "hf" else "") + "\n"
      f"- category: **{CATEGORY}** (synset `{SYNSET}`), one category only\n"
      f"- {NUM_TRAIN_SHAPES} training objects / {NUM_VAL_SHAPES} validation objects, "
      f"**strictly disjoint at object level** (asserted in §6)\n"
      f"- {NUM_POINTS} points per cloud, subsampled from 8192 by `{SUBSAMPLE}`\n"
      f"- per-point ShapeNet-Part labels ship with the data and were deliberately **never** given to "
      f"any model\n")

    # 3
    A("## 3. Preprocessing and normalisation\n")
    A("Repository procedure (`scripts/run_benchmark_kaggle.py :: make_entry`): centre at the centroid, "
      "divide by the maximum radius (unit sphere), subsampling first so the unit-sphere property is "
      "exact for the points used. Colour is mapped $C_0 = 2\\,RGB_{[0,1]}-1$. Raw 0–255 RGB is never "
      "diffused alongside normalised XYZ.\n")
    if norm is not None:
        A(df_to_md(norm[norm.tensor.str.contains("train", regex=False)]) + "\n")

    # 4
    A("## 4. Diffusion / noise setup\n")
    A(f"DDPM forward process only — no reverse sampler is ever run. `T = {T}`, `{BETA_SCHEDULE}` beta "
      f"schedule, $\\alpha_t=1-\\beta_t$, $\\bar\\alpha_t=\\prod_{{s\\le t}}\\alpha_s$, "
      f"$t\\sim\\mathrm{{Uniform}}\\{{0..{T-1}\\}}$ during training. Experiment 1 noises geometry only; "
      f"Experiment 2 noises colour only. Clean-signal recovery uses "
      f"$\\hat x_0=(x_t-\\sqrt{{1-\\bar\\alpha_t}}\\hat\\epsilon)/\\sqrt{{\\bar\\alpha_t}}$, clamped "
      f"to $[-1,1]$ (valid for both modalities by construction) and applied identically to both models "
      f"of a pair.\n")
    if snrt is not None:
        A("Effective SNR per modality at the evaluated timesteps — the same $t$ is a *harder* "
          "corruption for the lower-variance modality, which is why absolute losses are comparable "
          "**within** a pair only:\n")
        A(df_to_md(snrt) + "\n")

    # 5
    A("## 5. The four models\n")
    A("| model | input | target | conditioning |\n|---|---|---|---|\n"
      "| **G**   | $G_t$ | $\\epsilon_g$ | — |\n"
      "| **G+C** | $[G_t, C_0]$ | $\\epsilon_g$ | clean colour |\n"
      "| **C**   | $C_t$ | $\\epsilon_c$ | — |\n"
      "| **C+G** | $[C_t, G_0]$ | $\\epsilon_c$ | clean geometry |\n")
    A(f"One class (`PointDenoiser`): shared per-point MLP → {N_BLOCKS} residual blocks combining the "
      f"per-point feature with a permutation-invariant global max-pool, each FiLM-modulated by a "
      f"sinusoidal timestep embedding → per-point head. Width {WIDTH}. `[B,N,D] → [B,N,3]`.\n")
    if pars is not None:
        A(df_to_md(pars[["model", "input", "in_channels", "local_backbone", "params"]]) + "\n")
        A("A pair differs by one `Linear(3→width)` — the conditioned model is not a bigger model.\n")
    A("**Colour is an attribute, never a coordinate.** The default backbone is strictly pointwise, so "
      "no neighbourhood of any kind exists and a mixed XYZ+RGB distance is unrepresentable. "
      + ("The optional EdgeConv branch was enabled for the geometry pair; its kNN graph is built from "
         "$G_t$, which both geometry models hold, so no colour enters any neighbourhood and the pair "
         "stays fair.\n" if USE_LOCAL_GEOMETRY_BACKBONE else
         "The optional EdgeConv branch was left off.\n"))
    if leak is not None:
        A("Leakage test re-run on the **trained** weights: with $x_t$ held fixed, perturbing a "
          "modality a model does not receive must move its output by exactly 0, and perturbing one "
          "it does receive must move it:\n")
        A(df_to_md(leak) + "\n")

    # 6
    A("## 6. Training configuration\n")
    A(f"- AdamW, lr {LEARNING_RATE} ({LR_SCHEDULE} schedule), weight decay {WEIGHT_DECAY}, "
      f"batch size {BATCH_SIZE}, {NUM_EPOCHS} epochs, device `{DEVICE}`, seed {SEED}\n"
      f"- loss: plain noise-prediction MSE, $\\|\\epsilon-\\hat\\epsilon\\|_2^2$. No Chamfer, EMD, "
      f"perceptual, smoothness or part loss — the conditioning effect is deliberately isolated.\n"
      f"- **paired training**: both models of a pair see identical batches, identical sampled $t$ and "
      f"identical noise, drawn from generators seeded by (seed, epoch, step)\n"
      f"- per-epoch checkpoints in `{CKPT_DIR}`, resumable\n")
    if san is not None:
        A(f"Tiny-set overfit sanity check (§12): "
          f"{'**passed** for all four models' if bool(san.passed.all()) else '**FAILED** for ' + str(san.loc[~san.passed, 'model'].tolist())}"
          f" (reference: predicting $\\epsilon=0$ scores exactly 1.0).\n")
        A(df_to_md(san[["model", "loss_start", "loss_end", "ratio", "eps_t50", "eps_t750",
                        "x0_gain_t250", "passed"]]) + "\n")
        A("`x0_gain_t250` is the fraction of the naive “assume it is not noisy” "
          "reconstruction error the model removes at $t=250$; it is the criterion that distinguishes "
          "real denoising from echoing the input, which the $\\epsilon$-MSE average cannot.\n")

    # 7
    A("## 7. Evaluation metrics\n")
    A("Both models of a pair are scored by the same code on the same objects at the same timesteps "
      "from the **same noise tensor**.\n\n"
      "*Geometry (↓):* `eps_mse` the training objective; `x0_mse` / `x0_p2p` error of the recovered "
      "clean shape; `chamfer_l1/l2` correspondence-free shape fit; `colored_chamfer_dE` the ΔE76 "
      "between each predicted point's colour and its nearest GT point's — i.e. *did geometry error "
      "push points across colour boundaries*, the failure that actually matters for colored "
      "completion.\n\n"
      "*Colour:* `rgb_mae`/`rgb_mse`/`psnr` pointwise; `deltaE76` (the repo's existing metric) and "
      "`deltaE00` (CIEDE2000, added because CIE76 over-weights saturated colours) perceptual; "
      "`local_consistency_absdiff`/`_ratio` compares local colour roughness over XYZ kNN "
      "neighbourhoods — catches speckle (ratio ≫ 1) and over-smoothing (ratio ≪ 1); `hist_chi2` and "
      "`swd_lab` are position-blind distribution distances; `region_dE` averages ΔE76 between "
      "cell-mean colours over an XYZ-only FPS-Voronoi partition — catches *right palette, wrong "
      "place*, and `region_dE_norm` normalises it by a colour-scrambled control so that **0 = right "
      "colours in the right regions and 1 = chance**; `colour_edge_iou` is the IoU of the colour-edge "
      "point sets under a GT-derived threshold, catching boundaries that were smoothed away or "
      "hallucinated in the wrong place.\n")

    # 8
    A("## 8. Quantitative geometry results\n")
    if geot is not None:
        A(df_to_md(geot) + "\n")
    A(f"- noise prediction: {verdict(g_eps, 'Model G', 'Model G+C')}\n")
    A(f"- clean-XYZ recovery: {verdict(g_x0, 'Model G', 'Model G+C')}\n")
    r = row("geometry", "colored_chamfer_dE")
    if r is not None:
        A(f"- colour-aware geometry error: {verdict(r, 'Model G', 'Model G+C')}\n")

    # 9
    A("## 9. Quantitative colour results\n")
    if colt is not None:
        A(df_to_md(colt) + "\n")
    A(f"- noise prediction: {verdict(c_eps, 'Model C', 'Model C+G')}\n")
    A(f"- perceptual ΔE00: {verdict(c_de, 'Model C', 'Model C+G')}\n")
    for m in ("region_dE", "region_dE_norm", "colour_edge_iou", "swd_lab",
              "local_consistency_absdiff"):
        r = row("colour", m)
        if r is not None:
            A(f"- `{m}`: {verdict(r, 'Model C', 'Model C+G')}\n")

    # 10
    A("## 10. Timestep-wise comparison\n")
    for pair, ts, mets in (("geometry", tsg, ["eps_mse", "x0_mse"]),
                           ("colour", tsc, ["eps_mse", "deltaE00"])):
        if ts is None:
            continue
        for m in mets:
            sub = ts[ts.metric == m].drop(columns=["metric"])
            A(f"**{pair} — `{m}` {arrow(m)}**\n")
            A(df_to_md(sub) + "\n")
            imp = sub["improvement_%"].values
            if len(imp) >= 2 and np.all(np.isfinite(imp)):
                h = max(1, len(imp) // 2)
                lo, hi = float(imp[:h].mean()), float(imp[-h:].mean())
                trend = ("grows with the noise level" if hi > lo + 1 else
                         "shrinks as the noise level grows" if lo > hi + 1 else
                         "is roughly flat across noise levels")
                peak = int(np.argmax(imp))
                extra = ("" if peak in (0, len(imp) - 1) else
                         f" It is largest in the middle of the range, at t={int(sub.t.iloc[peak])} "
                         f"({imp[peak]:+.1f}%), so the trend is not monotone.")
                A(f"The benefit of conditioning **{trend}** "
                  f"({imp[0]:+.1f}% at t={int(sub.t.iloc[0])} → {imp[-1]:+.1f}% at "
                  f"t={int(sub.t.iloc[-1])}).{extra}\n")

    # 11
    A("## 11. Saved qualitative visualisations\n")
    figs = sorted(f for f in os.listdir(FIG_DIR) if f.endswith((".png", ".html")))
    for f in figs:
        A(f"- `{os.path.join(FIG_DIR, f)}`\n")

    # 12
    A("## 12. Best / median / worst validation examples\n")
    if bmc is not None:
        A(f"Colour (ranked by Model C+G ΔE00 at t={VIS_T}) — "
          f"`21_colour_best_median_worst.png`:\n")
        A(df_to_md(bmc) + "\n")
    if bmg is not None:
        A(f"Geometry (ranked by Model G+C x0-MSE at t={VIS_T}) — "
          f"`21_geometry_best_median_worst.png`:\n")
        A(df_to_md(bmg) + "\n")

    # 13 / 14
    A("## 13. Observations — geometry denoising\n")
    if g_eps is not None:
        if g_win and resolved(g_eps):
            A("Under this controlled denoising setup, **clean colour provides useful predictive "
              "information for recovering corrupted geometry**. It does *not* follow that a full joint "
              "diffusion completion framework outperforms a geometry-only completion system; that "
              "requires a completion experiment, not a denoising one.\n")
        elif g_win:
            A("The conditioned geometry model is ahead on average, but the bootstrap interval spans "
              "zero — this is a *direction*, not yet an effect. More validation objects, more noise "
              "draws or longer training are needed before it can be called evidence.\n")
        else:
            A("Clean colour did **not** help geometry denoising here. See §15 for candidate causes "
              "before treating this as a property of the modalities.\n")
    A("## 14. Observations — colour denoising\n")
    if c_eps is not None:
        if c_win and resolved(c_eps):
            A("**Geometry provides useful information for recovering corrupted colour** under this "
              "setup. The plausible mechanism is that within one category the canonical pose makes "
              "position predictive of material, so $G_0$ narrows the colour prior per point — which "
              "is what the region-wise and placement metrics probe.\n")
        elif c_win:
            A("The conditioned colour model leads on average but the interval spans zero; treat it as "
              "a direction, not a result.\n")
        else:
            A("Clean geometry did **not** help colour denoising here.\n")

    # 15
    A("## 15. Does the evidence support Colour→Geometry, Geometry→Colour, both, or neither?\n")
    gr, cr = resolved(g_eps) and g_win, resolved(c_eps) and c_win
    if gr and cr:
        A("**Both directions.** This is preliminary evidence for a **bidirectional geometry–colour "
          "coupling** in the denoising setting, which is the precondition the joint 6-D model needs. "
          "It remains preliminary: one category, one architecture, denoising rather than completion.\n")
    elif cr and not gr:
        A("**Geometry → Colour only.** The coupling is *asymmetric* here: geometry informs colour, but "
          "clean colour does not measurably help geometry. The practical reading is that the eventual "
          "model may benefit more from **directional conditioning** (geometry conditioning the colour "
          "branch) than from symmetric coupling — cheaper and easier to train than full joint "
          "denoising.\n")
    elif gr and not cr:
        A("**Colour → Geometry only.** The asymmetry runs opposite to the expected direction, which is "
          "worth investigating: it suggests colour is acting as a shape-identity cue rather than "
          "geometry acting as a colour prior.\n")
    else:
        A("**Neither direction is resolved.** This does **not** establish that the hypothesis is "
          "false. Before concluding anything about the modalities, rule out: (i) normalisation and "
          "the resulting per-modality SNR mismatch (§4 table); (ii) model capacity — a pointwise "
          "backbone can only exploit *marginal* relations $p(x_i\\mid c_i)$ and cannot represent "
          "neighbourhood structure; (iii) timestep conditioning; (iv) colour informativeness of this "
          "category (§2 reports the distinct-colour counts); (v) the point sampling; (vi) simply too "
          "few objects for the effect size present.\n")

    # 16
    A("## 16. Where metrics disagree\n")
    if agree is not None:
        A(df_to_md(agree) + "\n")
        favs = agree.favours.unique().tolist()
        if len(favs) > 1:
            split = {f: agree[agree.favours == f].metric.tolist() for f in favs}
            A("The colour metrics **do not agree**: " + "; ".join(
                f"`{', '.join(v)}` favour **{k}**" for k, v in split.items()) + ". ")
            posb = set(split.get(PAIRS['colour'][1], []))
            spatial = {"region_dE", "region_dE_norm", "colour_edge_iou"}
            if {"hist_chi2", "swd_lab"} & posb and not (spatial & posb):
                A("Specifically the *position-blind* distribution metrics favour the conditioned model "
                  "while the *spatial* ones do not — the signature of **right palette, wrong place**: "
                  "the model reproduces the colour distribution without putting each colour on the "
                  "correct geometric region. Trust the spatial metrics for the coupling question.\n")
            elif spatial & posb and not ({"hist_chi2", "swd_lab"} & posb):
                A("The *spatial* metrics favour the conditioned model while the distribution metrics "
                  "do not — colours are landing in the right regions even though the global palette "
                  "is no better. For the geometry↔colour coupling question this is the more "
                  "meaningful of the two.\n")
            else:
                A("Inspect §22's error maps before weighting any single metric.\n")
        else:
            A(f"All colour metrics point the same way (towards **{favs[0]}**), so no metric conflict "
              f"needs adjudicating. Confirm against the §22 error maps regardless — agreement among "
              f"aggregates is not the same as agreement with what the clouds look like.\n")
    A("A caution the numbers cannot enforce: ΔE alone is not sufficient. Read it jointly with the "
      "pointwise, spatial, distributional and qualitative evidence.\n")

    # 17
    A("## 17. Limitations\n")
    A(f"1. **Denoising, not completion.** Nothing is ever sampled; there is no reverse process, no "
      f"mask, no partial input. A win here is a necessary, not sufficient, condition for the joint "
      f"completion model.\n"
      f"2. **The clean auxiliary modality is a gift the real task does not give.** In completion the "
      f"conditioning modality is itself missing exactly where the target is. This experiment "
      f"deliberately measures the *upper bound* on how much that information could be worth.\n"
      f"3. **One category ({CATEGORY}), {NUM_TRAIN_SHAPES}+{NUM_VAL_SHAPES} objects, "
      f"{NUM_POINTS} points.** Effect sizes may not transfer across categories.\n"
      f"4. **One small pointwise architecture.** It can express marginal $p(\\text{{modality}}_i\\mid"
      f"\\text{{other}}_i)$ relations and a global shape/palette code, but not local neighbourhood "
      f"structure. A negative result is as much a statement about this backbone as about the data.\n"
      f"5. **Single seed.** Run-to-run variance is not estimated; the reported intervals are over "
      f"validation objects, not over training runs.\n"
      f"6. **The $[-1,1]$ clamp on $\\hat x_0$** dominates the high-$t$ reconstruction metrics for "
      f"both models equally; `eps_mse` is the clamp-free number.\n"
      f"7. Perfect CUDA determinism is not guaranteed; paired fairness comes from explicitly seeded "
      f"generators, which is checkable, rather than from global RNG state.\n")

    # 18
    A("## 18. The next minimal experiment\n")
    if cr and not gr:
        nxt = ("Keep the asymmetry and exploit it. Next: **replace clean $G_0$ with *noisy* $G_t$** in "
               "Model C+G and sweep the geometry noise level independently of the colour one. That "
               "measures how fast the geometry→colour benefit decays as the conditioning modality "
               "degrades — which is precisely what happens in completion, where geometry is itself "
               "uncertain. If the benefit survives moderate geometry noise, a geometry-conditioned "
               "colour branch is the cheapest useful step toward the joint model.")
    elif gr and cr:
        nxt = ("Both directions pay, so the next question is whether they pay *simultaneously*. Next: "
               "**one shared network denoising both modalities at once** — noise geometry and colour "
               "with independent timesteps $(t_g, t_c)$, predict $[\\epsilon_g,\\epsilon_c]$ jointly, "
               "and compare against the two separate conditioned models on this same validation set. "
               "That is the smallest experiment that is genuinely *joint* rather than two conditional "
               "models, and the independent timesteps let you measure whether coupling survives when "
               "both modalities are degraded together.")
    else:
        nxt = ("Before scaling anything, isolate whether the null is about the data or about the "
               "backbone. Next: **re-run the same four models with the local (EdgeConv) backbone "
               "enabled for the geometry pair** (`USE_LOCAL_GEOMETRY_BACKBONE = True`, kNN on $G_t$ "
               "so the pair stays fair) and, separately, on a more strongly textured category. If a "
               "local backbone or a more colourful category flips the result, the null was about "
               "capacity or colour informativeness — not about the modalities.")
    A(nxt + "\n")
    A("In all cases the step **after** that is the same: move from denoising to a masked setting — "
      "condition on a *partial* auxiliary modality — because that, not clean conditioning, is the "
      "regime the completion model actually lives in.\n")

    A("\n---\n")
    A(f"*Config: seed {SEED}, category {CATEGORY}, {NUM_POINTS} pts, T={T} ({BETA_SCHEDULE}), "
      f"width {WIDTH}×{N_BLOCKS} blocks, {NUM_EPOCHS} epochs, "
      f"eval t={EVAL_TIMESTEPS} × {EVAL_REPEATS} noise draws.*\n")
    return "\n".join(L)


REPORT_MD = generate_report()
report_path = os.path.join(RESULTS_DIR, "experiment_report.md")
with open(report_path, "w") as f:
    f.write(REPORT_MD)
print("report written to", report_path, f"({len(REPORT_MD)} chars)")
print("\n" + "=" * 78 + "\n")
print(REPORT_MD[:4000])
print("\n... (full text in the file above) ...")

In [ ]:
# ---- compact HTML twin of the report (no external markdown dependency) ----
import re as _re


def _inline(s):
    import html as _h
    s = _h.escape(s)
    s = _re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", s)
    s = _re.sub(r"`(.+?)`", r"<code>\1</code>", s)
    return s


def md_to_html(text):
    out, in_tbl = [], False
    for ln in text.split("\n"):
        s = ln.rstrip()
        if s.startswith("|") and s.endswith("|"):
            cells = [c.strip() for c in s.strip("|").split("|")]
            if set("".join(cells)) <= set("-: "):
                continue
            tag = "td" if in_tbl else "th"
            if not in_tbl:
                out.append("<table>"); in_tbl = True
            out.append("<tr>" + "".join(f"<{tag}>{_inline(c)}</{tag}>" for c in cells) + "</tr>")
            continue
        if in_tbl:
            out.append("</table>"); in_tbl = False
        if s.startswith("### "):
            out.append(f"<h3>{_inline(s[4:])}</h3>")
        elif s.startswith("## "):
            out.append(f"<h2>{_inline(s[3:])}</h2>")
        elif s.startswith("# "):
            out.append(f"<h1>{_inline(s[2:])}</h1>")
        elif s.startswith("> "):
            out.append(f"<blockquote>{_inline(s[2:])}</blockquote>")
        elif s.startswith("- "):
            out.append(f"<li>{_inline(s[2:])}</li>")
        elif s.strip() == "---":
            out.append("<hr>")
        elif s.strip() == "":
            out.append("")
        else:
            out.append(f"<p>{_inline(s)}</p>")
    if in_tbl:
        out.append("</table>")
    body = "\n".join(out)
    return ("<!doctype html><meta charset='utf-8'><title>Geometry-Colour feasibility report</title>"
            "<style>body{font:14px/1.6 -apple-system,Segoe UI,Roboto,sans-serif;max-width:900px;"
            "margin:2rem auto;padding:0 1rem;color:#222}table{border-collapse:collapse;margin:1rem 0;"
            "font-size:12px;display:block;overflow-x:auto}th,td{border:1px solid #ddd;padding:4px 8px;"
            "text-align:right}th{background:#f4f4f4}td:first-child,th:first-child{text-align:left}"
            "h1,h2{border-bottom:1px solid #eee;padding-bottom:.2em}blockquote{border-left:3px solid "
            "#e0574c;margin:1em 0;padding:.2em 1em;background:#fff6f5}</style>" + body)


html_path = os.path.join(RESULTS_DIR, "experiment_report.html")
with open(html_path, "w") as f:
    f.write(md_to_html(REPORT_MD))
print("html report ->", html_path)

print("\n" + "=" * 78)
print("SAVED OUTPUTS")
print("=" * 78)
for label, d in (("report", RESULTS_DIR), ("checkpoints", CKPT_DIR), ("history", HIST_DIR),
                 ("tables", TAB_DIR), ("figures", FIG_DIR)):
    files = sorted(f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f)))
    print(f"\n{label}  ({d})")
    for f in files:
        print(f"   {f:52s} {os.path.getsize(os.path.join(d, f))/1024:8.1f} KB")

## 25 · Next-step recommendation

Section 18 of the generated report already names the next minimal experiment, chosen from the actual
outcome. The cell below re-prints it next to the levers that matter most if you want to strengthen or
challenge the result first.

In [ ]:
print("=" * 78)
print("NEXT STEP — as chosen by this run's outcome (see §18 of the report)")
print("=" * 78)
if "REPORT_MD" in dir():
    seg = REPORT_MD.split("## 18. The next minimal experiment")
    print(seg[-1].split("\n---")[0].strip() if len(seg) > 1 else "(report not generated)")

print("\n" + "=" * 78)
print("IF YOU WANT TO STRENGTHEN THIS RESULT FIRST — cheapest levers, in order")
print("=" * 78)
print('''
 1. More validation objects (NUM_VAL_SHAPES 20 -> 50) and more noise draws (EVAL_REPEATS 4 -> 8).
    The bootstrap intervals shrink as sqrt(n); this is the cheapest way to resolve a borderline effect.
 2. Repeat over 3 seeds (SEED = 0,1,2) and report the spread. Single-seed intervals cover variation
    across objects, not across training runs -- the report says so explicitly, and that is the gap.
 3. Repeat on a second category (CATEGORY = "airplane" / "car"). If the sign flips, the effect is a
    property of chairs, not of the modalities.
 4. Raise capacity (WIDTH 128 -> 256, N_BLOCKS 3 -> 4) for BOTH models of a pair. A null result under
    a small pointwise backbone is partly a statement about the backbone.
 5. USE_LOCAL_GEOMETRY_BACKBONE = True for the geometry pair only (kNN on G_t, fair for both models,
    no colour in any neighbourhood). This is the one architectural change that tests whether local
    structure -- not just marginal per-point relations -- is where the coupling lives.

 Deliberately NOT next: a reverse sampler, cross-attention, part conditioning, or a full completion
 system. Each adds an error source; this study exists to keep them separate.
''')
print("results directory:", RESULTS_DIR)